In [ ]:
import os
import sys
import json
import glob
import time
import logging
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import librosa
import librosa.display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("pipeline")

@dataclass
class Paths:
    cv_root: str = r"C:\Users\Acer\Downloads\1781705453816-cv-corpus-26.0-2026-06-12-gju\cv-corpus-26.0-2026-06-12\gju"
    unrated_root: str = r"C:\Users\Acer\Downloads\unrated_transcripts\unrated_transcripts"
    massive_root: str = r"C:\Users\Acer\Downloads\amazon-massive-dataset-1.0"
    chime_trans: str = r"C:\Users\Acer\Downloads\CHiME6_transcriptions\transcriptions"
    chime_floor: str = r"C:\Users\Acer\Downloads\CHiME6_floorplans"
    output_dir: str = "pipeline_outputs"

    def check_all(self) -> Dict[str, bool]:
        status = {}
        for name, p in vars(self).items():
            if name == "output_dir":
                continue
            exists = os.path.exists(p)
            status[name] = exists
            log.info(f"{name:15s} -> {'FOUND' if exists else 'MISSING'} :: {p}")
        return status


PATHS = Paths()
os.makedirs(PATHS.output_dir, exist_ok=True)
STATUS = PATHS.check_all()


def savefig(fig, name):
    out = os.path.join(PATHS.output_dir, name)
    fig.savefig(out, dpi=150, bbox_inches="tight")
    log.info(f"Saved figure -> {out}")

class CommonVoiceCorpus:

    def __init__(self, root: str):
        self.root = root
        self.tables: Dict[str, pd.DataFrame] = {}
        self.clips_dir = os.path.join(root, "clips")

    def load_tsvs(self):
        if not os.path.isdir(self.root):
            log.warning("Common Voice root missing, skipping.")
            return
        for tsv_path in glob.glob(os.path.join(self.root, "*.tsv")):
            name = Path(tsv_path).stem
            try:
                df = pd.read_csv(tsv_path, sep="\t", low_memory=False)
                self.tables[name] = df
                log.info(f"  loaded {name}: {df.shape}")
            except Exception as e:
                log.error(f"  failed to load {tsv_path}: {e}")

    def get(self, name: str) -> Optional[pd.DataFrame]:
        return self.tables.get(name)

    def summarize(self) -> pd.DataFrame:
        rows = []
        for name, df in self.tables.items():
            rows.append({
                "table": name,
                "rows": len(df),
                "cols": df.shape[1],
                "columns": ", ".join(df.columns[:8]),
            })
        return pd.DataFrame(rows)

    def plot_metadata_overview(self):
        validated = self.get("validated")
        if validated is None:
            log.warning("No validated.tsv, skipping metadata plot.")
            return

        fig = plt.figure(figsize=(18, 10))
        gs = gridspec.GridSpec(2, 3, figure=fig)

        # Sentence length
        ax0 = fig.add_subplot(gs[0, 0])
        lens = validated["sentence"].astype(str).str.len()
        ax0.hist(lens, bins=40, color="#4C72B0", edgecolor="white")
        ax0.set_title("Sentence length (characters)")
        ax0.set_xlabel("Length")
        ax0.set_ylabel("Count")

        # Word count
        ax1 = fig.add_subplot(gs[0, 1])
        wc = validated["sentence"].astype(str).str.split().apply(len)
        ax1.hist(wc, bins=30, color="#55A868", edgecolor="white")
        ax1.set_title("Sentence length (words)")
        ax1.set_xlabel("Word count")

        # Vote balance (validated vs rejected signal)
        ax2 = fig.add_subplot(gs[0, 2])
        if {"up_votes", "down_votes"}.issubset(validated.columns):
            ax2.scatter(validated["up_votes"], validated["down_votes"], alpha=0.3, s=10)
            ax2.set_xlabel("Up votes")
            ax2.set_ylabel("Down votes")
            ax2.set_title("Vote distribution")
        else:
            ax2.text(0.5, 0.5, "up_votes/down_votes\nnot present", ha="center")
            ax2.axis("off")

        # Gender
        ax3 = fig.add_subplot(gs[1, 0])
        if "gender" in validated.columns:
            validated["gender"].fillna("unspecified").value_counts().plot(
                kind="bar", ax=ax3, color="#C44E52"
            )
            ax3.set_title("Speaker gender")
        else:
            ax3.axis("off")
            ax3.text(0.5, 0.5, "no gender column", ha="center")

        # Age
        ax4 = fig.add_subplot(gs[1, 1])
        if "age" in validated.columns:
            validated["age"].fillna("unspecified").value_counts().plot(
                kind="bar", ax=ax4, color="#8172B2"
            )
            ax4.set_title("Speaker age bracket")
        else:
            ax4.axis("off")
            ax4.text(0.5, 0.5, "no age column", ha="center")

        # Unique speakers
        ax5 = fig.add_subplot(gs[1, 2])
        if "client_id" in validated.columns:
            n_speakers = validated["client_id"].nunique()
            clips_per_speaker = validated["client_id"].value_counts()
            ax5.hist(clips_per_speaker, bins=30, color="#CCB974", edgecolor="white")
            ax5.set_title(f"Clips per speaker (n={n_speakers} speakers)")
            ax5.set_xlabel("Clips")
        else:
            ax5.axis("off")

        fig.suptitle("Common Voice (Assamese) — Metadata Overview", fontsize=16)
        plt.tight_layout()
        savefig(fig, "01_cv_metadata_overview.pdf")
        plt.show()

    def sample_clip_paths(self, n=25, seed=42) -> List[str]:
        validated = self.get("validated")
        if validated is None or not os.path.isdir(self.clips_dir):
            return []
        sample = validated.sample(min(n, len(validated)), random_state=seed)
        paths = [os.path.join(self.clips_dir, p) for p in sample["path"]]
        existing = [p for p in paths if os.path.exists(p)]
        log.info(f"Sampled {len(existing)}/{len(paths)} clips that exist on disk.")
        return existing


# =====================================================================
# SECTION 2 — AUDIO FEATURE EXTRACTION (real signal processing)
# =====================================================================

class AudioFeatureExtractor:
    def __init__(self, sr: int = 16000, n_mfcc: int = 13):
        self.sr = sr
        self.n_mfcc = n_mfcc

    def extract(self, path: str) -> Dict[str, Any]:
        y, sr = librosa.load(path, sr=self.sr)
        duration = librosa.get_duration(y=y, sr=sr)

        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=self.n_mfcc)
        delta = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)

        centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
        zcr = librosa.feature.zero_crossing_rate(y)[0]
        rms = librosa.feature.rms(y=y)[0]
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)

        f0, voiced_flag, voiced_probs = librosa.pyin(
            y, fmin=librosa.note_to_hz("C2"), fmax=librosa.note_to_hz("C7")
        )

        return {
            "path": path,
            "duration_s": duration,
            "y": y,
            "sr": sr,
            "mfcc": mfcc,
            "delta": delta,
            "delta2": delta2,
            "centroid_mean": np.nanmean(centroid),
            "bandwidth_mean": np.nanmean(bandwidth),
            "rolloff_mean": np.nanmean(rolloff),
            "zcr_mean": np.nanmean(zcr),
            "rms_mean": np.nanmean(rms),
            "chroma_mean": np.nanmean(chroma),
            "f0_mean": np.nanmean(f0) if f0 is not None else np.nan,
            "voiced_fraction": np.nanmean(voiced_flag.astype(float)) if voiced_flag is not None else np.nan,
        }

    def batch_extract(self, paths: List[str]) -> pd.DataFrame:
        rows = []
        for p in paths:
            try:
                feats = self.extract(p)
                row = {k: v for k, v in feats.items() if k not in ("y", "mfcc", "delta", "delta2")}
                rows.append(row)
            except Exception as e:
                log.error(f"Failed on {p}: {e}")
        return pd.DataFrame(rows)

    def plot_single_clip(self, path: str, save_name: str = "02_mfcc_single_clip.pdf"):
        feats = self.extract(path)
        y, sr = feats["y"], feats["sr"]

        fig, axes = plt.subplots(5, 1, figsize=(12, 16))

        librosa.display.waveshow(y, sr=sr, ax=axes[0])
        axes[0].set_title(f"Waveform — {os.path.basename(path)} ({feats['duration_s']:.2f}s)")

        S = librosa.feature.melspectrogram(y=y, sr=sr)
        S_db = librosa.power_to_db(S, ref=np.max)
        img0 = librosa.display.specshow(S_db, sr=sr, x_axis="time", y_axis="mel", ax=axes[1])
        axes[1].set_title("Mel spectrogram (dB)")
        fig.colorbar(img0, ax=axes[1], format="%+2.0f dB")

        img1 = librosa.display.specshow(feats["mfcc"], x_axis="time", ax=axes[2])
        axes[2].set_title(f"MFCC ({self.n_mfcc} coefficients)")
        fig.colorbar(img1, ax=axes[2])

        img2 = librosa.display.specshow(feats["delta"], x_axis="time", ax=axes[3])
        axes[3].set_title("Delta MFCC")
        fig.colorbar(img2, ax=axes[3])

        img3 = librosa.display.specshow(feats["delta2"], x_axis="time", ax=axes[4])
        axes[4].set_title("Delta-Delta MFCC")
        fig.colorbar(img3, ax=axes[4])

        plt.tight_layout()
        savefig(fig, save_name)
        plt.show()

    def plot_feature_distributions(self, features_df: pd.DataFrame):
        numeric_cols = [
            "duration_s", "centroid_mean", "bandwidth_mean",
            "rolloff_mean", "zcr_mean", "rms_mean", "chroma_mean", "f0_mean",
        ]
        numeric_cols = [c for c in numeric_cols if c in features_df.columns]

        fig, axes = plt.subplots(2, 4, figsize=(20, 9))
        axes = axes.flatten()
        for i, col in enumerate(numeric_cols):
            axes[i].hist(features_df[col].dropna(), bins=20, color="#4C72B0", edgecolor="white")
            axes[i].set_title(col)
        for j in range(len(numeric_cols), len(axes)):
            axes[j].axis("off")

        fig.suptitle("Acoustic Feature Distributions Across Sampled Clips", fontsize=16)
        plt.tight_layout()
        savefig(fig, "03_feature_distributions.pdf")
        plt.show()

    def plot_feature_correlations(self, features_df: pd.DataFrame):
        numeric_cols = [
            "duration_s", "centroid_mean", "bandwidth_mean",
            "rolloff_mean", "zcr_mean", "rms_mean", "chroma_mean", "f0_mean",
        ]
        numeric_cols = [c for c in numeric_cols if c in features_df.columns]
        corr = features_df[numeric_cols].corr()

        fig, ax = plt.subplots(figsize=(9, 7))
        sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
        ax.set_title("Correlation Between Acoustic Features")
        plt.tight_layout()
        savefig(fig, "04_feature_correlation_heatmap.pdf")
        plt.show()


# =====================================================================
# SECTION 3 — ASR BASELINE (real model, real errors)
# =====================================================================

class ASRBaseline:

    def __init__(self, model_size: str = "small", language: str = "as"):
        self.language = language
        self.model = None
        try:
            import whisper
            self.model = whisper.load_model(model_size)
            log.info(f"Loaded Whisper '{model_size}' model.")
        except Exception as e:
            log.error(f"Could not load Whisper: {e}")

    def transcribe(self, path: str) -> Optional[str]:
        if self.model is None:
            return None
        try:
            result = self.model.transcribe(path, language=self.language)
            return result["text"].strip()
        except Exception as e:
            log.error(f"Transcription failed on {path}: {e}")
            return None

    def evaluate(self, cv: CommonVoiceCorpus, paths: List[str]) -> pd.DataFrame:
        try:
            from jiwer import wer, cer
        except ImportError:
            log.error("jiwer not installed — cannot compute WER/CER. `pip install jiwer`")
            return pd.DataFrame()

        validated = cv.get("validated")
        if validated is None:
            return pd.DataFrame()

        ref_lookup = validated.set_index("path")["sentence"].to_dict()

        rows = []
        for path in paths:
            basename = os.path.basename(path)
            ref = ref_lookup.get(basename)
            if ref is None:
                continue
            hyp = self.transcribe(path)
            if hyp is None:
                continue
            rows.append({
                "path": basename,
                "reference": ref,
                "hypothesis": hyp,
                "wer": wer(ref, hyp),
                "cer": cer(ref, hyp),
                "ref_len_chars": len(ref),
                "ref_len_words": len(ref.split()),
            })
        return pd.DataFrame(rows)

    def plot_results(self, results_df: pd.DataFrame):
        if results_df.empty:
            log.warning("No ASR results to plot.")
            return

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        axes[0, 0].hist(results_df["wer"], bins=15, color="#C44E52", edgecolor="white")
        axes[0, 0].axvline(results_df["wer"].mean(), color="black", linestyle="--",
                            label=f"mean={results_df['wer'].mean():.3f}")
        axes[0, 0].set_title("WER distribution")
        axes[0, 0].legend()

        axes[0, 1].hist(results_df["cer"], bins=15, color="#55A868", edgecolor="white")
        axes[0, 1].axvline(results_df["cer"].mean(), color="black", linestyle="--",
                            label=f"mean={results_df['cer'].mean():.3f}")
        axes[0, 1].set_title("CER distribution")
        axes[0, 1].legend()

        axes[1, 0].scatter(results_df["ref_len_words"], results_df["wer"], alpha=0.6)
        axes[1, 0].set_xlabel("Reference length (words)")
        axes[1, 0].set_ylabel("WER")
        axes[1, 0].set_title("WER vs. utterance length")

        sorted_df = results_df.sort_values("wer")
        axes[1, 1].barh(range(len(sorted_df)), sorted_df["wer"], color="#8172B2")
        axes[1, 1].set_yticks([])
        axes[1, 1].set_xlabel("WER")
        axes[1, 1].set_title("Per-clip WER, sorted")

        fig.suptitle(f"Whisper ASR Baseline — Assamese ({len(results_df)} clips)", fontsize=15)
        plt.tight_layout()
        savefig(fig, "05_asr_baseline_results.pdf")
        plt.show()


# =====================================================================
# SECTION 4 — UNRATED TRANSCRIPTS (schema unknown, inspect first)
# =====================================================================

class UnratedTranscripts:

    def __init__(self, root: str):
        self.root = root
        self.file_index: Dict[str, List[str]] = {}

    def scan(self):
        if not os.path.isdir(self.root):
            log.warning("Unrated transcripts root missing, skipping.")
            return
        for ext in ["txt", "json", "csv", "tsv"]:
            files = glob.glob(os.path.join(self.root, "**", f"*.{ext}"), recursive=True)
            if files:
                self.file_index[ext] = files
                log.info(f"  found {len(files)} .{ext} files")

    def preview(self, n=3):
        for ext, files in self.file_index.items():
            print(f"\n--- Preview of .{ext} files ---")
            for f in files[:n]:
                print(f"\n[{f}]")
                try:
                    if ext == "json":
                        with open(f, "r", encoding="utf-8") as fh:
                            data = json.load(fh)
                        print(json.dumps(data, indent=2)[:500])
                    else:
                        with open(f, "r", encoding="utf-8", errors="ignore") as fh:
                            print(fh.read()[:500])
                except Exception as e:
                    print(f"  couldn't preview: {e}")

    def basic_stats(self) -> pd.DataFrame:
        rows = []
        for ext, files in self.file_index.items():
            for f in files:
                try:
                    size_kb = os.path.getsize(f) / 1024
                    rows.append({"file": os.path.basename(f), "ext": ext, "size_kb": size_kb})
                except Exception:
                    continue
        return pd.DataFrame(rows)

    def plot_file_sizes(self, stats_df: pd.DataFrame):
        if stats_df.empty:
            return
        fig, ax = plt.subplots(figsize=(10, 5))
        for ext in stats_df["ext"].unique():
            subset = stats_df[stats_df["ext"] == ext]
            ax.hist(subset["size_kb"], bins=20, alpha=0.6, label=ext)
        ax.set_xlabel("File size (KB)")
        ax.set_ylabel("Count")
        ax.set_title("Unrated Transcripts — File Size Distribution by Type")
        ax.legend()
        plt.tight_layout()
        savefig(fig, "06_unrated_transcripts_filesizes.pdf")
        plt.show()


# =====================================================================
# SECTION 5 — AMAZON MASSIVE (text NLU: intents + slots, not ASR)
# =====================================================================

class MassiveCorpus:

    def __init__(self, root: str):
        self.root = root
        self.df: Optional[pd.DataFrame] = None
        self.locale_used: Optional[str] = None

    def load(self, preferred_locale: str = "en-US"):
        if not os.path.isdir(self.root):
            log.warning("MASSIVE root missing, skipping.")
            return
        files = glob.glob(os.path.join(self.root, "**", "*.jsonl"), recursive=True)
        if not files:
            log.warning("No .jsonl files found under MASSIVE root.")
            return

        target = None
        for f in files:
            if preferred_locale in os.path.basename(f):
                target = f
                break
        target = target or files[0]
        self.locale_used = os.path.basename(target)

        records = []
        with open(target, "r", encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        self.df = pd.DataFrame(records)
        log.info(f"Loaded MASSIVE file: {target} -> {self.df.shape}")

    def plot_overview(self):
        if self.df is None:
            return
        fig = plt.figure(figsize=(18, 10))
        gs = gridspec.GridSpec(2, 2, figure=fig)

        ax0 = fig.add_subplot(gs[0, 0])
        if "intent" in self.df.columns:
            self.df["intent"].value_counts().head(15).plot(kind="barh", ax=ax0, color="#4C72B0")
            ax0.invert_yaxis()
            ax0.set_title("Top 15 intents")

        ax1 = fig.add_subplot(gs[0, 1])
        if "utt" in self.df.columns:
            lens = self.df["utt"].astype(str).str.split().apply(len)
            ax1.hist(lens, bins=25, color="#55A868", edgecolor="white")
            ax1.set_title("Utterance length (words)")

        ax2 = fig.add_subplot(gs[1, 0])
        if "domain" in self.df.columns:
            self.df["domain"].value_counts().plot(kind="bar", ax=ax2, color="#C44E52")
            ax2.set_title("Domain distribution")
        elif "scenario" in self.df.columns:
            self.df["scenario"].value_counts().plot(kind="bar", ax=ax2, color="#C44E52")
            ax2.set_title("Scenario distribution")

        ax3 = fig.add_subplot(gs[1, 1])
        if "slots" in self.df.columns or "annot_utt" in self.df.columns:
            slot_col = "annot_utt" if "annot_utt" in self.df.columns else "slots"
            slot_tag_counts = self.df[slot_col].astype(str).str.count(r"\[")
            ax3.hist(slot_tag_counts, bins=range(0, slot_tag_counts.max() + 2), color="#8172B2")
            ax3.set_title("Slot tags per utterance")

        fig.suptitle(f"MASSIVE Overview — {self.locale_used}", fontsize=16)
        plt.tight_layout()
        savefig(fig, "07_massive_overview.pdf")
        plt.show()


# =====================================================================
# SECTION 6 — CHiME-6 (transcriptions + floorplans)
# =====================================================================

class Chime6Corpus:

    def __init__(self, trans_root: str, floor_root: str):
        self.trans_root = trans_root
        self.floor_root = floor_root
        self.segments_df: Optional[pd.DataFrame] = None

    def load_transcriptions(self):
        if not os.path.isdir(self.trans_root):
            log.warning("CHiME-6 transcriptions root missing, skipping.")
            return
        files = glob.glob(os.path.join(self.trans_root, "*.json"))
        if not files:
            log.warning("No .json transcription files found.")
            return

        all_segments = []
        for f in files:
            try:
                with open(f, "r", encoding="utf-8") as fh:
                    data = json.load(fh)
                if isinstance(data, list):
                    for seg in data:
                        seg["_session"] = os.path.basename(f)
                    all_segments.extend(data)
            except Exception as e:
                log.error(f"Failed to parse {f}: {e}")

        self.segments_df = pd.DataFrame(all_segments)
        log.info(f"Loaded {len(self.segments_df)} segments from {len(files)} sessions.")

    def plot_speaker_activity(self):
        if self.segments_df is None or self.segments_df.empty:
            return
        df = self.segments_df

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        if "speaker" in df.columns:
            df["speaker"].value_counts().plot(kind="bar", ax=axes[0], color="#4C72B0")
            axes[0].set_title("Turns per speaker (all sessions)")

        if {"start_time", "end_time"}.issubset(df.columns):
            def to_seconds(t):
                try:
                    parts = str(t).split(":")
                    return sum(float(p) * 60 ** i for i, p in enumerate(reversed(parts)))
                except Exception:
                    return np.nan

            df["duration"] = df["end_time"].apply(to_seconds) - df["start_time"].apply(to_seconds)
            axes[1].hist(df["duration"].dropna(), bins=30, color="#55A868", edgecolor="white")
            axes[1].set_title("Utterance duration distribution")
            axes[1].set_xlabel("Seconds")

        fig.suptitle("CHiME-6 Transcription Analysis", fontsize=15)
        plt.tight_layout()
        savefig(fig, "08_chime6_speaker_activity.pdf")
        plt.show()

    def list_floorplans(self):
        if not os.path.isdir(self.floor_root):
            log.warning("CHiME-6 floorplans root missing.")
            return []
        images = glob.glob(os.path.join(self.floor_root, "*"))
        log.info(f"Found {len(images)} floorplan files.")
        return images


# =====================================================================
# SECTION 7 — CROSS-DATASET COMPARISON
# =====================================================================

def cross_dataset_summary(cv_features: pd.DataFrame, asr_results: pd.DataFrame):

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if not cv_features.empty:
        axes[0].boxplot(
            [cv_features["duration_s"].dropna(),
             cv_features["f0_mean"].dropna()],
            labels=["Clip duration (s)", "Mean F0 (Hz)"]
        )
        axes[0].set_title("Common Voice acoustic summary")
    else:
        axes[0].text(0.5, 0.5, "No CV audio features available", ha="center")
        axes[0].axis("off")

    if not asr_results.empty:
        axes[1].boxplot(
            [asr_results["wer"], asr_results["cer"]],
            labels=["WER", "CER"]
        )
        axes[1].set_title("ASR baseline error summary")
    else:
        axes[1].text(0.5, 0.5, "No ASR results available", ha="center")
        axes[1].axis("off")

    plt.tight_layout()
    savefig(fig, "09_cross_dataset_summary.pdf")
    plt.show()


# =====================================================================
# MAIN
# =====================================================================

def main():
    log.info("=== Starting pipeline ===")

    # --- Common Voice ---
    cv = CommonVoiceCorpus(PATHS.cv_root)
    cv.load_tsvs()
    print(cv.summarize())
    cv.plot_metadata_overview()

    sample_paths = cv.sample_clip_paths(n=25)

    extractor = AudioFeatureExtractor()
    cv_features = pd.DataFrame()
    if sample_paths:
        extractor.plot_single_clip(sample_paths[0])
        cv_features = extractor.batch_extract(sample_paths)
        extractor.plot_feature_distributions(cv_features)
        extractor.plot_feature_correlations(cv_features)
        cv_features.to_csv(os.path.join(PATHS.output_dir, "cv_audio_features.csv"), index=False)

    # --- ASR baseline ---
    asr_results = pd.DataFrame()
    if sample_paths:
        asr = ASRBaseline(model_size="small", language="as")
        asr_results = asr.evaluate(cv, sample_paths)
        asr.plot_results(asr_results)
        if not asr_results.empty:
            asr_results.to_csv(os.path.join(PATHS.output_dir, "asr_results.csv"), index=False)

    # --- Unrated transcripts ---
    unrated = UnratedTranscripts(PATHS.unrated_root)
    unrated.scan()
    unrated.preview(n=2)
    stats_df = unrated.basic_stats()
    unrated.plot_file_sizes(stats_df)

    # --- MASSIVE ---
    massive = MassiveCorpus(PATHS.massive_root)
    massive.load()
    massive.plot_overview()

    # --- CHiME-6 ---
    chime = Chime6Corpus(PATHS.chime_trans, PATHS.chime_floor)
    chime.load_transcriptions()
    chime.plot_speaker_activity()
    chime.list_floorplans()

    # --- Cross-dataset ---
    cross_dataset_summary(cv_features, asr_results)

    log.info("=== Pipeline complete. Check the pipeline_outputs/ folder. ===")


if __name__ == "__main__":
    main()

In [ ]:

!pip install transformers
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import librosa
from jiwer import wer, cer
import pandas as pd
import numpy as np


class Wav2Vec2Baseline:
    def __init__(self, model_name="facebook/wav2vec2-base-960h", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.processor = Wav2Vec2Processor.from_pretrained(model_name)
        self.model = Wav2Vec2ForCTC.from_pretrained(model_name).to(self.device)
        self.model.eval()
        self.model_name = model_name

    def transcribe(self, path: str, sr: int = 16000) -> str:
        y, _ = librosa.load(path, sr=sr)
        inputs = self.processor(y, sampling_rate=sr, return_tensors="pt", padding=True)
        input_values = inputs.input_values.to(self.device)

        with torch.no_grad():
            logits = self.model(input_values).logits

        pred_ids = torch.argmax(logits, dim=-1)
        text = self.processor.batch_decode(pred_ids)[0]
        return text.strip()

    def evaluate(self, cv_corpus, paths: list) -> pd.DataFrame:
        validated = cv_corpus.get("validated")
        if validated is None:
            return pd.DataFrame()

        ref_lookup = validated.set_index("path")["sentence"].to_dict()
        rows = []
        for path in paths:
            basename = os.path.basename(path)
            ref = ref_lookup.get(basename)
            if ref is None:
                continue
            try:
                hyp = self.transcribe(path)
            except Exception as e:
                log.error(f"wav2vec2 failed on {path}: {e}")
                continue
            rows.append({
                "path": basename,
                "reference": ref,
                "hypothesis": hyp,
                "wer": wer(ref, hyp),
                "cer": cer(ref, hyp),
                "model": self.model_name,
            })
        return pd.DataFrame(rows)


def compare_asr_models(whisper_results: pd.DataFrame, wav2vec_results: pd.DataFrame) -> pd.DataFrame:
    whisper_results = whisper_results.copy()
    whisper_results["model"] = "whisper-small"

    combined = pd.concat([whisper_results, wav2vec_results], ignore_index=True)

    summary = combined.groupby("model").agg(
        mean_wer=("wer", "mean"),
        median_wer=("wer", "median"),
        std_wer=("wer", "std"),
        mean_cer=("cer", "mean"),
        n_clips=("wer", "count"),
    ).reset_index()

    return summary, combined


def plot_model_comparison(summary_df: pd.DataFrame, combined_df: pd.DataFrame):
    if summary_df.empty:
        log.warning("Nothing to compare — check that both models actually ran.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].bar(summary_df["model"], summary_df["mean_wer"],
                yerr=summary_df["std_wer"], capsize=6, color=["#4C72B0", "#C44E52"])
    axes[0].set_title("Mean WER by model (± std)")
    axes[0].set_ylabel("WER")

    for model in combined_df["model"].unique():
        subset = combined_df[combined_df["model"] == model]
        axes[1].hist(subset["wer"], bins=15, alpha=0.6, label=model)
    axes[1].set_title("WER distribution overlay")
    axes[1].legend()

    # paired comparison per clip, where both models have a result
    pivot = combined_df.pivot_table(index="path", columns="model", values="wer")
    pivot = pivot.dropna()
    if not pivot.empty and pivot.shape[1] == 2:
        cols = pivot.columns.tolist()
        axes[2].scatter(pivot[cols[0]], pivot[cols[1]], alpha=0.6)
        lims = [0, max(pivot.max().max(), 1.0)]
        axes[2].plot(lims, lims, "k--", alpha=0.4)
        axes[2].set_xlabel(cols[0])
        axes[2].set_ylabel(cols[1])
        axes[2].set_title("Per-clip WER: model A vs model B")
    else:
        axes[2].text(0.5, 0.5, "Not enough paired data\nfor per-clip comparison", ha="center")
        axes[2].axis("off")

    fig.suptitle("Whisper vs. wav2vec2 — Head-to-Head on Assamese", fontsize=15)
    plt.tight_layout()
    savefig(fig, "10_asr_model_comparison.pdf")
    plt.show()

    print("\n=== Comparison Summary ===")
    print(summary_df.to_string(index=False))

def build_demographic_wer_table(asr_results: pd.DataFrame, cv_corpus) -> pd.DataFrame:
    validated = cv_corpus.get("validated")
    if validated is None or asr_results.empty:
        return pd.DataFrame()

    needed_cols = {"path", "client_id", "age", "gender"}
    missing = needed_cols - set(validated.columns)
    if missing:
        log.warning(f"Missing demographic columns: {missing}. Can't do this breakdown.")
        return pd.DataFrame()

    meta = validated[["path", "client_id", "age", "gender"]].copy()
    merged = asr_results.merge(meta, left_on="path", right_on="path", how="left")
    merged["age"] = merged["age"].fillna("unspecified")
    merged["gender"] = merged["gender"].fillna("unspecified")
    return merged


def plot_demographic_wer(merged_df: pd.DataFrame):
    if merged_df.empty:
        log.warning("No demographic-linked WER data to plot.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(16, 11))

    # WER by gender - boxplot, not just means, so you can see spread
    gender_groups = [merged_df[merged_df["gender"] == g]["wer"].values
                      for g in merged_df["gender"].unique()]
    axes[0, 0].boxplot(gender_groups, labels=merged_df["gender"].unique())
    axes[0, 0].set_title("WER by gender")
    axes[0, 0].set_ylabel("WER")
    axes[0, 0].tick_params(axis="x", rotation=30)

    # WER by age bracket
    age_order = sorted(merged_df["age"].unique())
    age_groups = [merged_df[merged_df["age"] == a]["wer"].values for a in age_order]
    axes[0, 1].boxplot(age_groups, labels=age_order)
    axes[0, 1].set_title("WER by age bracket")
    axes[0, 1].tick_params(axis="x", rotation=45)

    # sample size per group — this matters, small groups make noisy stats
    group_counts = merged_df.groupby(["gender", "age"]).size().reset_index(name="n")
    pivot_counts = group_counts.pivot(index="age", columns="gender", values="n").fillna(0)
    sns.heatmap(pivot_counts, annot=True, fmt=".0f", cmap="Blues", ax=axes[1, 0])
    axes[1, 0].set_title("Sample size per age × gender group")

    # mean WER heatmap, same grid — so you can eyeball where the gaps are
    pivot_wer = merged_df.groupby(["gender", "age"])["wer"].mean().reset_index()
    pivot_wer = pivot_wer.pivot(index="age", columns="gender", values="wer")
    sns.heatmap(pivot_wer, annot=True, fmt=".3f", cmap="Reds", ax=axes[1, 1])
    axes[1, 1].set_title("Mean WER per age × gender group")

    fig.suptitle("ASR Error Rate Broken Down by Speaker Demographics", fontsize=15)
    plt.tight_layout()
    savefig(fig, "11_demographic_wer_breakdown.pdf")
    plt.show()

    # print a real statistical check instead of eyeballing it
    from scipy import stats
    if merged_df["gender"].nunique() >= 2:
        groups = [g["wer"].values for _, g in merged_df.groupby("gender") if len(g) > 1]
        if len(groups) >= 2:
            f_stat, p_val = stats.f_oneway(*groups)
            print(f"One-way ANOVA across gender groups: F={f_stat:.3f}, p={p_val:.4f}")
            if p_val < 0.05:
                print("  -> statistically significant difference. Worth digging into why.")
            else:
                print("  -> not significant at p<0.05. Could just be sample-size noise.")


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score


class MassiveIntentClassifier:

    def __init__(self, max_features=5000, min_df=2):
        self.vectorizer = TfidfVectorizer(max_features=max_features, min_df=min_df, ngram_range=(1, 2))
        self.clf = LogisticRegression(max_iter=1000, multi_class="auto")
        self.label_names = None

    def fit(self, texts, labels):
        X = self.vectorizer.fit_transform(texts)
        self.label_names = sorted(set(labels))
        self.clf.fit(X, labels)

    def predict(self, texts):
        X = self.vectorizer.transform(texts)
        return self.clf.predict(X)

    def evaluate(self, texts, labels):
        preds = self.predict(texts)
        acc = accuracy_score(labels, preds)
        report = classification_report(labels, preds, zero_division=0)
        cm = confusion_matrix(labels, preds, labels=self.label_names)
        return acc, report, cm


def run_intent_classification(massive_df: pd.DataFrame):
    if massive_df is None or massive_df.empty:
        log.warning("No MASSIVE data loaded, skipping classifier.")
        return None, None, None

    text_col = "utt" if "utt" in massive_df.columns else None
    label_col = "intent" if "intent" in massive_df.columns else None

    if text_col is None or label_col is None:
        log.warning(f"Couldn't find utterance/intent columns. Got: {massive_df.columns.tolist()}")
        return None, None, None

    df = massive_df[[text_col, label_col]].dropna()

    # drop intents with too few examples — can't stratify-split a class of 1
    counts = df[label_col].value_counts()
    valid_labels = counts[counts >= 5].index
    df = df[df[label_col].isin(valid_labels)]

    if df[label_col].nunique() < 2:
        log.warning("Not enough distinct intents with sufficient examples.")
        return None, None, None

    X_train, X_test, y_train, y_test = train_test_split(
        df[text_col], df[label_col], test_size=0.2, random_state=42, stratify=df[label_col]
    )

    clf = MassiveIntentClassifier()
    clf.fit(X_train, y_train)
    acc, report, cm = clf.evaluate(X_test, y_test)

    print(f"\n=== Intent Classifier Accuracy: {acc:.4f} ===")
    print(report)

    return clf, cm, acc


def plot_confusion_matrix(cm: np.ndarray, label_names: list, top_n: int = 15):
    if cm is None:
        return

    support = cm.sum(axis=1)
    top_idx = np.argsort(support)[::-1][:top_n]
    top_idx = sorted(top_idx)

    cm_sub = cm[np.ix_(top_idx, top_idx)]
    labels_sub = [label_names[i] for i in top_idx]

    # normalize by row so it shows recall-style percentages, easier to read
    cm_norm = cm_sub.astype(float) / (cm_sub.sum(axis=1, keepdims=True) + 1e-9)

    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=labels_sub, yticklabels=labels_sub, ax=ax, cbar_kws={"label": "Recall"})
    ax.set_xlabel("Predicted intent")
    ax.set_ylabel("True intent")
    ax.set_title(f"Confusion Matrix — Top {top_n} Intents (row-normalized)")
    plt.xticks(rotation=60, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    savefig(fig, "12_massive_confusion_matrix.pdf")
    plt.show()


def plot_per_intent_performance(y_test, y_pred, label_names):
    from sklearn.metrics import precision_recall_fscore_support

    precision, recall, f1, support = precision_recall_fscore_support(
        y_test, y_pred, labels=label_names, zero_division=0
    )

    perf_df = pd.DataFrame({
        "intent": label_names,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
    }).sort_values("support", ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(12, 7))
    x = np.arange(len(perf_df))
    width = 0.25
    ax.bar(x - width, perf_df["precision"], width, label="Precision")
    ax.bar(x, perf_df["recall"], width, label="Recall")
    ax.bar(x + width, perf_df["f1"], width, label="F1")
    ax.set_xticks(x)
    ax.set_xticklabels(perf_df["intent"], rotation=60, ha="right")
    ax.set_title("Per-Intent Precision / Recall / F1 (top 15 by support)")
    ax.legend()
    plt.tight_layout()
    savefig(fig, "13_massive_per_intent_scores.pdf")
    plt.show()


def parse_time_to_seconds(t):

    try:
        return float(t)
    except (ValueError, TypeError):
        pass
    try:
        parts = str(t).split(":")
        parts = [float(p) for p in parts]
        seconds = 0
        for i, p in enumerate(reversed(parts)):
            seconds += p * (60 ** i)
        return seconds
    except Exception:
        return np.nan


def build_session_timelines(chime_trans_root: str) -> dict:
    files = glob.glob(os.path.join(chime_trans_root, "*.json"))
    sessions = {}

    for f in files:
        try:
            with open(f, "r", encoding="utf-8") as fh:
                data = json.load(fh)
        except Exception as e:
            log.error(f"Couldn't parse {f}: {e}")
            continue

        if not isinstance(data, list):
            continue

        rows = []
        for seg in data:
            speaker = seg.get("speaker", "unknown")
            start = parse_time_to_seconds(seg.get("start_time"))
            end = parse_time_to_seconds(seg.get("end_time"))
            if np.isnan(start) or np.isnan(end):
                continue
            rows.append({"speaker": speaker, "start": start, "end": end, "duration": end - start})

        if rows:
            session_name = os.path.splitext(os.path.basename(f))[0]
            sessions[session_name] = pd.DataFrame(rows)

    log.info(f"Built timelines for {len(sessions)} sessions.")
    return sessions


def plot_session_timelines(sessions: dict, max_sessions: int = 6):
    if not sessions:
        log.warning("No session timeline data available.")
        return

    session_names = list(sessions.keys())[:max_sessions]
    n = len(session_names)
    fig, axes = plt.subplots(n, 1, figsize=(16, 3 * n), sharex=False)

    if n == 1:
        axes = [axes]

    palette = sns.color_palette("tab10")

    for ax, name in zip(axes, session_names):
        df = sessions[name]
        speakers = sorted(df["speaker"].unique())
        speaker_y = {s: i for i, s in enumerate(speakers)}
        colors = {s: palette[i % len(palette)] for i, s in enumerate(speakers)}

        for _, row in df.iterrows():
            y = speaker_y[row["speaker"]]
            ax.barh(y, row["duration"], left=row["start"], height=0.6,
                    color=colors[row["speaker"]], edgecolor="none", alpha=0.85)

        ax.set_yticks(list(speaker_y.values()))
        ax.set_yticklabels(list(speaker_y.keys()))
        ax.set_xlabel("Time (s)")
        ax.set_title(f"Session: {name}  ({len(df)} turns, {len(speakers)} speakers)")

    fig.suptitle("CHiME-6 Speaker-Turn Timelines by Session", fontsize=16, y=1.01)
    plt.tight_layout()
    savefig(fig, "14_chime6_session_timelines.pdf")
    plt.show()


def plot_turn_taking_stats(sessions: dict):

    if not sessions:
        return

    overlap_counts = []
    gap_stats = []

    for name, df in sessions.items():
        df_sorted = df.sort_values("start").reset_index(drop=True)
        overlaps = 0
        gaps = []
        for i in range(len(df_sorted) - 1):
            curr_end = df_sorted.loc[i, "end"]
            next_start = df_sorted.loc[i + 1, "start"]
            gap = next_start - curr_end
            if gap < 0:
                overlaps += 1
            else:
                gaps.append(gap)
        overlap_counts.append({"session": name, "overlaps": overlaps, "total_turns": len(df_sorted)})
        if gaps:
            gap_stats.append({"session": name, "mean_gap": np.mean(gaps), "median_gap": np.median(gaps)})

    overlap_df = pd.DataFrame(overlap_counts)
    gap_df = pd.DataFrame(gap_stats)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if not overlap_df.empty:
        overlap_df["overlap_rate"] = overlap_df["overlaps"] / overlap_df["total_turns"].clip(lower=1)
        axes[0].bar(overlap_df["session"], overlap_df["overlap_rate"], color="#C44E52")
        axes[0].set_title("Speaker overlap rate per session")
        axes[0].set_ylabel("Fraction of turns that overlap the previous one")
        axes[0].tick_params(axis="x", rotation=45)

    if not gap_df.empty:
        axes[1].bar(gap_df["session"], gap_df["mean_gap"], color="#4C72B0")
        axes[1].set_title("Mean inter-turn gap per session")
        axes[1].set_ylabel("Seconds")
        axes[1].tick_params(axis="x", rotation=45)

    fig.suptitle("Turn-Taking Dynamics Across CHiME-6 Sessions", fontsize=15)
    plt.tight_layout()
    savefig(fig, "15_chime6_turn_taking_dynamics.pdf")
    plt.show()

    print("\n=== Overlap Summary ===")
    print(overlap_df.to_string(index=False) if not overlap_df.empty else "No data.")


def main():
    log.info("=== Starting pipeline ===")

    # --- Common Voice ---
    cv = CommonVoiceCorpus(PATHS.cv_root)
    cv.load_tsvs()
    print(cv.summarize())
    cv.plot_metadata_overview()

    sample_paths = cv.sample_clip_paths(n=25)

    extractor = AudioFeatureExtractor()
    cv_features = pd.DataFrame()
    if sample_paths:
        extractor.plot_single_clip(sample_paths[0])
        cv_features = extractor.batch_extract(sample_paths)
        extractor.plot_feature_distributions(cv_features)
        extractor.plot_feature_correlations(cv_features)
        cv_features.to_csv(os.path.join(PATHS.output_dir, "cv_audio_features.csv"), index=False)

    # --- Whisper baseline ---
    asr_results = pd.DataFrame()
    if sample_paths:
        asr = ASRBaseline(model_size="small", language="as")
        asr_results = asr.evaluate(cv, sample_paths)
        asr.plot_results(asr_results)
        if not asr_results.empty:
            asr_results.to_csv(os.path.join(PATHS.output_dir, "asr_results.csv"), index=False)

    # --- Unrated transcripts ---
    unrated = UnratedTranscripts(PATHS.unrated_root)
    unrated.scan()
    unrated.preview(n=2)
    stats_df = unrated.basic_stats()
    unrated.plot_file_sizes(stats_df)

    # --- MASSIVE ---
    massive = MassiveCorpus(PATHS.massive_root)
    massive.load()
    massive.plot_overview()

    # --- CHiME-6 ---
    chime = Chime6Corpus(PATHS.chime_trans, PATHS.chime_floor)
    chime.load_transcriptions()
    chime.plot_speaker_activity()
    chime.list_floorplans()

    cross_dataset_summary(cv_features, asr_results)

    # ===================================================
    # Extended analysis — same scope, so everything above
    # is still visible down here. No more NameError.
    # ===================================================

    # 1. wav2vec2 comparison
    wav2vec_results = pd.DataFrame()
    if sample_paths:
        try:
            w2v = Wav2Vec2Baseline()
            wav2vec_results = w2v.evaluate(cv, sample_paths)
        except Exception as e:
            log.error(f"wav2vec2 baseline failed to run: {e}")

    if not asr_results.empty and not wav2vec_results.empty:
        summary, combined = compare_asr_models(asr_results, wav2vec_results)
        plot_model_comparison(summary, combined)
    else:
        log.warning("Skipping model comparison — one or both baselines produced no results.")

    # 2. demographic WER
    demo_df = build_demographic_wer_table(asr_results, cv)
    plot_demographic_wer(demo_df)

    # 3. MASSIVE intent classifier
    clf, cm, acc = run_intent_classification(massive.df)
    if clf is not None:
        plot_confusion_matrix(cm, clf.label_names)

    # 4. CHiME-6 timelines
    sessions = build_session_timelines(PATHS.chime_trans)
    plot_session_timelines(sessions)
    plot_turn_taking_stats(sessions)

    log.info("=== Pipeline complete. Check the pipeline_outputs/ folder. ===")


if __name__ == "__main__":
    main()

In [ ]:
import os
import sys
import json
import glob
import time
import logging
import warnings
import itertools
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any, Tuple, Union
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap, Normalize
import seaborn as sns
import librosa
import librosa.display
import scipy
from scipy import stats, signal
from scipy.fft import fft, ifft
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("pipeline")


# =====================================================================
# SECTION 0 — CONFIG
# =====================================================================

@dataclass
class Paths:
    cv_root: str = r"C:\Users\Acer\Downloads\1781705453816-cv-corpus-26.0-2026-06-12-gju\cv-corpus-26.0-2026-06-12\gju"
    unrated_root: str = r"C:\Users\Acer\Downloads\unrated_transcripts\unrated_transcripts"
    massive_root: str = r"C:\Users\Acer\Downloads\amazon-massive-dataset-1.0"
    chime_trans: str = r"C:\Users\Acer\Downloads\CHiME6_transcriptions\transcriptions"
    chime_floor: str = r"C:\Users\Acer\Downloads\CHiME6_floorplans"
    output_dir: str = "pipeline_outputs"

    def check_all(self) -> Dict[str, bool]:
        status = {}
        for name, p in vars(self).items():
            if name == "output_dir":
                continue
            exists = os.path.exists(p)
            status[name] = exists
            log.info(f"{name:15s} -> {'FOUND' if exists else 'MISSING'} :: {p}")
        return status


PATHS = Paths()
os.makedirs(PATHS.output_dir, exist_ok=True)
STATUS = PATHS.check_all()


def savefig(fig, name):
    out = os.path.join(PATHS.output_dir, name)
    fig.savefig(out, dpi=150, bbox_inches="tight")
    log.info(f"Saved figure -> {out}")


# =====================================================================
# SECTION 1 — COMMON VOICE (ASSAMESE)
# =====================================================================

class CommonVoiceCorpus:
    def __init__(self, root: str):
        self.root = root
        self.tables: Dict[str, pd.DataFrame] = {}
        self.clips_dir = os.path.join(root, "clips")

    def load_tsvs(self):
        if not os.path.isdir(self.root):
            log.warning("Common Voice root missing, skipping.")
            return
        for tsv_path in glob.glob(os.path.join(self.root, "*.tsv")):
            name = Path(tsv_path).stem
            try:
                df = pd.read_csv(tsv_path, sep="\t", low_memory=False)
                self.tables[name] = df
                log.info(f"  loaded {name}: {df.shape}")
            except Exception as e:
                log.error(f"  failed to load {tsv_path}: {e}")

    def get(self, name: str) -> Optional[pd.DataFrame]:
        return self.tables.get(name)

    def summarize(self) -> pd.DataFrame:
        rows = []
        for name, df in self.tables.items():
            rows.append({
                "table": name,
                "rows": len(df),
                "cols": df.shape[1],
                "columns": ", ".join(df.columns[:8]),
            })
        return pd.DataFrame(rows)

    def plot_metadata_overview(self):
        validated = self.get("validated")
        if validated is None:
            log.warning("No validated.tsv, skipping metadata plot.")
            return

        fig = plt.figure(figsize=(18, 10))
        gs = gridspec.GridSpec(2, 3, figure=fig)

        # Sentence length
        ax0 = fig.add_subplot(gs[0, 0])
        lens = validated["sentence"].astype(str).str.len()
        ax0.hist(lens, bins=40, color="#4C72B0", edgecolor="white")
        ax0.set_title("Sentence length (characters)")
        ax0.set_xlabel("Length")
        ax0.set_ylabel("Count")

        # Word count
        ax1 = fig.add_subplot(gs[0, 1])
        wc = validated["sentence"].astype(str).str.split().apply(len)
        ax1.hist(wc, bins=30, color="#55A868", edgecolor="white")
        ax1.set_title("Sentence length (words)")
        ax1.set_xlabel("Word count")

        # Vote balance (validated vs rejected signal)
        ax2 = fig.add_subplot(gs[0, 2])
        if {"up_votes", "down_votes"}.issubset(validated.columns):
            ax2.scatter(validated["up_votes"], validated["down_votes"], alpha=0.3, s=10)
            ax2.set_xlabel("Up votes")
            ax2.set_ylabel("Down votes")
            ax2.set_title("Vote distribution")
        else:
            ax2.text(0.5, 0.5, "up_votes/down_votes\nnot present", ha="center")
            ax2.axis("off")

        # Gender
        ax3 = fig.add_subplot(gs[1, 0])
        if "gender" in validated.columns:
            validated["gender"].fillna("unspecified").value_counts().plot(
                kind="bar", ax=ax3, color="#C44E52"
            )
            ax3.set_title("Speaker gender")
        else:
            ax3.axis("off")
            ax3.text(0.5, 0.5, "no gender column", ha="center")

        # Age
        ax4 = fig.add_subplot(gs[1, 1])
        if "age" in validated.columns:
            validated["age"].fillna("unspecified").value_counts().plot(
                kind="bar", ax=ax4, color="#8172B2"
            )
            ax4.set_title("Speaker age bracket")
        else:
            ax4.axis("off")
            ax4.text(0.5, 0.5, "no age column", ha="center")

        # Unique speakers
        ax5 = fig.add_subplot(gs[1, 2])
        if "client_id" in validated.columns:
            n_speakers = validated["client_id"].nunique()
            clips_per_speaker = validated["client_id"].value_counts()
            ax5.hist(clips_per_speaker, bins=30, color="#CCB974", edgecolor="white")
            ax5.set_title(f"Clips per speaker (n={n_speakers} speakers)")
            ax5.set_xlabel("Clips")
        else:
            ax5.axis("off")

        fig.suptitle("Common Voice (Assamese) — Metadata Overview", fontsize=16)
        plt.tight_layout()
        savefig(fig, "01_cv_metadata_overview.pdf")
        plt.show()

    def sample_clip_paths(self, n=25, seed=42) -> List[str]:
        validated = self.get("validated")
        if validated is None or not os.path.isdir(self.clips_dir):
            return []
        sample = validated.sample(min(n, len(validated)), random_state=seed)
        paths = [os.path.join(self.clips_dir, p) for p in sample["path"]]
        existing = [p for p in paths if os.path.exists(p)]
        log.info(f"Sampled {len(existing)}/{len(paths)} clips that exist on disk.")
        return existing


# =====================================================================
# SECTION 2 — AUDIO FEATURE EXTRACTION (real signal processing)
# =====================================================================

class AdvancedAudioFeatureExtractor:

    def __init__(self, sr: int = 16000, n_mfcc: int = 13, n_formants: int = 4):
        self.sr = sr
        self.n_mfcc = n_mfcc
        self.n_formants = n_formants

    def extract(self, path: str) -> Dict[str, Any]:
        y, sr = librosa.load(path, sr=self.sr)
        duration = librosa.get_duration(y=y, sr=sr)

        # Standard MFCC and derivatives
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=self.n_mfcc)
        delta = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)

        # Spectral features
        centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
        flatness = librosa.feature.spectral_flatness(y=y)[0]
        entropy = self._spectral_entropy(y, sr)

        # Energy / RMS
        rms = librosa.feature.rms(y=y)[0]

        # Zero crossing rate
        zcr = librosa.feature.zero_crossing_rate(y)[0]

        # Chroma
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)

        # Pitch (F0) using PYIN
        f0, voiced_flag, voiced_probs = librosa.pyin(
            y, fmin=librosa.note_to_hz("C2"), fmax=librosa.note_to_hz("C7")
        )
        f0_mean = np.nanmean(f0) if f0 is not None else np.nan
        voiced_fraction = np.nanmean(voiced_flag.astype(float)) if voiced_flag is not None else np.nan

        # Formants (using LPC)
        formants = self._extract_formants(y, sr, self.n_formants)

        # Harmonic-to-noise ratio (HNR)
        hnr = self._compute_hnr(y, sr)

        # Temporal features: speaking rate (estimated from envelope)
        envelope = np.abs(y)
        env_peaks, _ = signal.find_peaks(envelope, height=0.1*np.max(envelope), distance=int(0.1*sr))
        speaking_rate = len(env_peaks) / duration if duration > 0 else 0

        return {
            "path": path,
            "duration_s": duration,
            "y": y,
            "sr": sr,
            "mfcc": mfcc,
            "delta": delta,
            "delta2": delta2,
            "centroid_mean": np.nanmean(centroid),
            "bandwidth_mean": np.nanmean(bandwidth),
            "rolloff_mean": np.nanmean(rolloff),
            "flatness_mean": np.nanmean(flatness),
            "entropy_mean": np.nanmean(entropy),
            "rms_mean": np.nanmean(rms),
            "zcr_mean": np.nanmean(zcr),
            "chroma_mean": np.nanmean(chroma),
            "f0_mean": f0_mean,
            "voiced_fraction": voiced_fraction,
            "formants": formants.tolist() if formants is not None else [np.nan]*self.n_formants,
            "hnr_mean": np.nanmean(hnr) if hnr is not None else np.nan,
            "speaking_rate": speaking_rate,
        }

    def _spectral_entropy(self, y, sr):
        S = np.abs(librosa.stft(y))**2
        S_norm = S / (S.sum(axis=0) + 1e-12)
        entropy = -np.sum(S_norm * np.log(S_norm + 1e-12), axis=0)
        return entropy

    def _extract_formants(self, y, sr, n_formants):
        try:
            # Pre-emphasis
            y_emph = librosa.effects.preemphasis(y)
            # LPC order: typically 2 + 2 * n_formants, but we use 2*n_formants + 2
            order = 2 * n_formants + 2
            # Compute LPC coefficients
            lpc_coeffs = librosa.lpc(y_emph, order=order)
            # Roots of the LPC polynomial
            roots = np.roots(lpc_coeffs)
            # Keep only roots with positive imaginary part and within unit circle
            roots = roots[np.imag(roots) >= 0]
            roots = roots[np.abs(roots) < 1.0]
            # Compute angles and convert to frequencies
            angles = np.angle(roots)
            freqs = angles * sr / (2 * np.pi)
            # Sort by magnitude (bandwidth) or just keep n largest peaks
            # We pick those with largest magnitude (most prominent)
            mags = np.abs(roots)
            idx = np.argsort(mags)[::-1]
            # Return top n_formants
            formants = freqs[idx][:n_formants]
            # Ensure we have n_formants values
            while len(formants) < n_formants:
                formants = np.append(formants, np.nan)
            return formants[:n_formants]
        except Exception:
            return np.array([np.nan] * n_formants)

    def _compute_hnr(self, y, sr):
        try:
            # Use librosa's yin for pitch, then compute HNR
            f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=50, fmax=500)
            # Compute HNR from autocorrelation
            # Simple implementation: use scipy.signal.periodogram
            # More robust: use autocorrelation of the residual after removing harmonics
            # For brevity, return placeholder
            # Actually, we can compute HNR using the ratio of harmonic energy to noise energy
            # Use a simple method: compute power spectrum, find harmonic peaks and interpolate noise floor
            return np.array([0.0])  # placeholder
        except:
            return np.array([0.0])

    def batch_extract(self, paths: List[str]) -> pd.DataFrame:
        rows = []
        for p in paths:
            try:
                feats = self.extract(p)
                row = {k: v for k, v in feats.items() if k not in ("y", "mfcc", "delta", "delta2")}
                rows.append(row)
            except Exception as e:
                log.error(f"Failed on {p}: {e}")
        return pd.DataFrame(rows)

    def plot_single_clip(self, path: str, save_name: str = "02_mfcc_single_clip.pdf"):
        feats = self.extract(path)
        y, sr = feats["y"], feats["sr"]

        fig = plt.figure(figsize=(15, 16))
        gs = gridspec.GridSpec(5, 2, figure=fig, height_ratios=[1, 1, 1, 1, 0.8])

        # Waveform
        ax0 = fig.add_subplot(gs[0, 0])
        librosa.display.waveshow(y, sr=sr, ax=ax0)
        ax0.set_title(f"Waveform — {os.path.basename(path)} ({feats['duration_s']:.2f}s)")

        # Spectrogram
        ax1 = fig.add_subplot(gs[0, 1])
        S = librosa.feature.melspectrogram(y=y, sr=sr)
        S_db = librosa.power_to_db(S, ref=np.max)
        img1 = librosa.display.specshow(S_db, sr=sr, x_axis="time", y_axis="mel", ax=ax1)
        ax1.set_title("Mel spectrogram (dB)")
        fig.colorbar(img1, ax=ax1, format="%+2.0f dB")

        # MFCC
        ax2 = fig.add_subplot(gs[1, 0])
        img2 = librosa.display.specshow(feats["mfcc"], x_axis="time", ax=ax2)
        ax2.set_title(f"MFCC ({self.n_mfcc} coefficients)")
        fig.colorbar(img2, ax=ax2)

        # Delta MFCC
        ax3 = fig.add_subplot(gs[1, 1])
        img3 = librosa.display.specshow(feats["delta"], x_axis="time", ax=ax3)
        ax3.set_title("Delta MFCC")
        fig.colorbar(img3, ax=ax3)

        # Delta-Delta MFCC
        ax4 = fig.add_subplot(gs[2, 0])
        img4 = librosa.display.specshow(feats["delta2"], x_axis="time", ax=ax4)
        ax4.set_title("Delta-Delta MFCC")
        fig.colorbar(img4, ax=ax4)

        # Chroma
        ax5 = fig.add_subplot(gs[2, 1])
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        img5 = librosa.display.specshow(chroma, x_axis="time", y_axis="chroma", ax=ax5)
        ax5.set_title("Chroma")
        fig.colorbar(img5, ax=ax5)

        # Pitch contour (F0)
        ax6 = fig.add_subplot(gs[3, 0])
        f0, voiced_flag, _ = librosa.pyin(y, fmin=librosa.note_to_hz("C2"), fmax=librosa.note_to_hz("C7"))
        times = librosa.times_like(f0, sr=sr)
        ax6.plot(times, f0, color='red', label='F0')
        ax6.set_xlabel("Time (s)")
        ax6.set_ylabel("Frequency (Hz)")
        ax6.set_title("Pitch contour (F0)")
        ax6.legend()

        # Formants (if available)
        ax7 = fig.add_subplot(gs[3, 1])
        formants = feats.get("formants", [np.nan]*self.n_formants)
        if not np.isnan(formants).all():
            ax7.bar(range(1, len(formants)+1), formants, color='blue')
            ax7.set_xticks(range(1, len(formants)+1))
            ax7.set_ylabel("Frequency (Hz)")
            ax7.set_title("Formant frequencies")
        else:
            ax7.text(0.5, 0.5, "Formants not computed", ha="center")
            ax7.axis("off")

        # Summary statistics
        ax8 = fig.add_subplot(gs[4, :])
        stats_dict = {
            "Duration": f"{feats['duration_s']:.2f}s",
            "F0 mean": f"{feats['f0_mean']:.1f} Hz",
            "Voiced fraction": f"{feats['voiced_fraction']:.2f}",
            "Spectral centroid": f"{feats['centroid_mean']:.0f} Hz",
            "Spectral bandwidth": f"{feats['bandwidth_mean']:.0f} Hz",
            "RMS energy": f"{feats['rms_mean']:.4f}",
            "ZCR": f"{feats['zcr_mean']:.3f}",
            "Entropy": f"{feats['entropy_mean']:.2f}",
            "Speaking rate": f"{feats['speaking_rate']:.1f} peaks/s",
            "HNR": f"{feats.get('hnr_mean', np.nan):.2f} dB",
        }
        ax8.axis('off')
        ax8.table(cellText=[list(stats_dict.values())],
                  rowLabels=[""],
                  colLabels=list(stats_dict.keys()),
                  cellLoc='center', loc='center')

        fig.suptitle("Detailed Acoustic Analysis of One Clip", fontsize=16)
        plt.tight_layout()
        savefig(fig, save_name)
        plt.show()

    def plot_feature_distributions(self, features_df: pd.DataFrame):
        numeric_cols = [
            "duration_s", "centroid_mean", "bandwidth_mean",
            "rolloff_mean", "flatness_mean", "entropy_mean",
            "zcr_mean", "rms_mean", "chroma_mean", "f0_mean",
            "voiced_fraction", "hnr_mean", "speaking_rate"
        ]
        # Add formant columns if present
        formant_cols = [c for c in features_df.columns if c.startswith("formants_")]
        numeric_cols = [c for c in numeric_cols if c in features_df.columns] + formant_cols

        n = len(numeric_cols)
        ncols = 4
        nrows = (n + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(20, 5*nrows))
        axes = axes.flatten() if nrows > 1 else axes

        for i, col in enumerate(numeric_cols):
            axes[i].hist(features_df[col].dropna(), bins=20, color="#4C72B0", edgecolor="white")
            axes[i].set_title(col)
            axes[i].set_xlabel("Value")
            axes[i].set_ylabel("Count")
        for j in range(len(numeric_cols), len(axes)):
            axes[j].axis("off")

        fig.suptitle("Acoustic Feature Distributions Across Sampled Clips", fontsize=16)
        plt.tight_layout()
        savefig(fig, "03_feature_distributions.pdf")
        plt.show()

    def plot_feature_correlations(self, features_df: pd.DataFrame):
        numeric_cols = [
            "duration_s", "centroid_mean", "bandwidth_mean",
            "rolloff_mean", "flatness_mean", "entropy_mean",
            "zcr_mean", "rms_mean", "chroma_mean", "f0_mean",
            "voiced_fraction", "hnr_mean", "speaking_rate"
        ]
        formant_cols = [c for c in features_df.columns if c.startswith("formants_")]
        numeric_cols = [c for c in numeric_cols if c in features_df.columns] + formant_cols
        corr = features_df[numeric_cols].corr()

        fig, ax = plt.subplots(figsize=(12, 10))
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
                    square=True, linewidths=0.5, ax=ax)
        ax.set_title("Correlation Between Acoustic Features")
        plt.tight_layout()
        savefig(fig, "04_feature_correlation_heatmap.pdf")
        plt.show()


# =====================================================================
# SECTION 3 — ASR BASELINE (real model, real errors)
# =====================================================================

class ASRBaseline:

    def __init__(self, model_size: str = "small", language: str = "as"):
        self.language = language
        self.model = None
        try:
            import whisper
            self.model = whisper.load_model(model_size)
            log.info(f"Loaded Whisper '{model_size}' model.")
        except Exception as e:
            log.error(f"Could not load Whisper: {e}")

    def transcribe(self, path: str) -> Optional[str]:
        if self.model is None:
            return None
        try:
            result = self.model.transcribe(path, language=self.language)
            return result["text"].strip()
        except Exception as e:
            log.error(f"Transcription failed on {path}: {e}")
            return None

    def evaluate(self, cv: CommonVoiceCorpus, paths: List[str]) -> pd.DataFrame:
        try:
            from jiwer import wer, cer
        except ImportError:
            log.error("jiwer not installed — cannot compute WER/CER. `pip install jiwer`")
            return pd.DataFrame()

        validated = cv.get("validated")
        if validated is None:
            return pd.DataFrame()

        ref_lookup = validated.set_index("path")["sentence"].to_dict()

        rows = []
        for path in paths:
            basename = os.path.basename(path)
            ref = ref_lookup.get(basename)
            if ref is None:
                continue
            hyp = self.transcribe(path)
            if hyp is None:
                continue
            rows.append({
                "path": basename,
                "reference": ref,
                "hypothesis": hyp,
                "wer": wer(ref, hyp),
                "cer": cer(ref, hyp),
                "ref_len_chars": len(ref),
                "ref_len_words": len(ref.split()),
            })
        return pd.DataFrame(rows)

    def plot_results(self, results_df: pd.DataFrame):
        if results_df.empty:
            log.warning("No ASR results to plot.")
            return

        fig = plt.figure(figsize=(18, 12))
        gs = gridspec.GridSpec(3, 3, figure=fig)

        # WER distribution
        ax0 = fig.add_subplot(gs[0, 0])
        ax0.hist(results_df["wer"], bins=15, color="#C44E52", edgecolor="white")
        ax0.axvline(results_df["wer"].mean(), color="black", linestyle="--",
                    label=f"mean={results_df['wer'].mean():.3f}")
        ax0.set_title("WER distribution")
        ax0.legend()

        # CER distribution
        ax1 = fig.add_subplot(gs[0, 1])
        ax1.hist(results_df["cer"], bins=15, color="#55A868", edgecolor="white")
        ax1.axvline(results_df["cer"].mean(), color="black", linestyle="--",
                    label=f"mean={results_df['cer'].mean():.3f}")
        ax1.set_title("CER distribution")
        ax1.legend()

        # WER vs length
        ax2 = fig.add_subplot(gs[0, 2])
        ax2.scatter(results_df["ref_len_words"], results_df["wer"], alpha=0.6)
        ax2.set_xlabel("Reference length (words)")
        ax2.set_ylabel("WER")
        ax2.set_title("WER vs. utterance length")

        # Error types (insertions/deletions/substitutions) if we can compute
        # We'll compute using jiwer's measures
        try:
            from jiwer import compute_measures
            ins, dels, subs = [], [], []
            for _, row in results_df.iterrows():
                m = compute_measures(row["reference"], row["hypothesis"])
                ins.append(m["insertions"])
                dels.append(m["deletions"])
                subs.append(m["substitutions"])
            results_df["insertions"] = ins
            results_df["deletions"] = dels
            results_df["substitutions"] = subs
            ax3 = fig.add_subplot(gs[1, 0])
            ax3.boxplot([results_df["insertions"], results_df["deletions"], results_df["substitutions"]],
                        labels=["Ins", "Del", "Sub"])
            ax3.set_title("Error type distribution")
        except:
            pass

        # Histogram of WER by clip
        ax4 = fig.add_subplot(gs[1, 1])
        sorted_df = results_df.sort_values("wer")
        ax4.barh(range(len(sorted_df)), sorted_df["wer"], color="#8172B2")
        ax4.set_yticks([])
        ax4.set_xlabel("WER")
        ax4.set_title("Per-clip WER, sorted")

        # Scatter of WER vs CER
        ax5 = fig.add_subplot(gs[1, 2])
        ax5.scatter(results_df["wer"], results_df["cer"], alpha=0.6)
        ax5.set_xlabel("WER")
        ax5.set_ylabel("CER")
        ax5.set_title("WER vs CER")

        # Error examples: worst and best
        ax6 = fig.add_subplot(gs[2, 0])
        worst = results_df.nlargest(5, "wer")
        best = results_df.nsmallest(5, "wer")
        examples = pd.concat([best, worst])
        ax6.axis('off')
        ax6.table(cellText=examples[["path", "wer", "reference", "hypothesis"]].values,
                  colLabels=["Path", "WER", "Ref", "Hyp"],
                  cellLoc='left', loc='center')

        # Cumulative distribution of WER
        ax7 = fig.add_subplot(gs[2, 1])
        sorted_wer = np.sort(results_df["wer"])
        yvals = np.arange(1, len(sorted_wer)+1) / len(sorted_wer)
        ax7.plot(sorted_wer, yvals, marker='.', linestyle='none')
        ax7.set_xlabel("WER")
        ax7.set_ylabel("CDF")
        ax7.set_title("CDF of WER")

        # Boxplot of WER by speaker (if speaker info available)
        ax8 = fig.add_subplot(gs[2, 2])
        # We need speaker mapping; we don't have it. So skip.

        fig.suptitle(f"Whisper ASR Baseline — Assamese ({len(results_df)} clips)", fontsize=15)
        plt.tight_layout()
        savefig(fig, "05_asr_baseline_results.pdf")
        plt.show()


# =====================================================================
# SECTION 4 — UNRATED TRANSCRIPTS (schema unknown, inspect first)
# =====================================================================

class UnratedTranscripts:

    def __init__(self, root: str):
        self.root = root
        self.file_index: Dict[str, List[str]] = {}

    def scan(self):
        if not os.path.isdir(self.root):
            log.warning("Unrated transcripts root missing, skipping.")
            return
        for ext in ["txt", "json", "csv", "tsv"]:
            files = glob.glob(os.path.join(self.root, "**", f"*.{ext}"), recursive=True)
            if files:
                self.file_index[ext] = files
                log.info(f"  found {len(files)} .{ext} files")

    def preview(self, n=3):
        for ext, files in self.file_index.items():
            print(f"\n--- Preview of .{ext} files ---")
            for f in files[:n]:
                print(f"\n[{f}]")
                try:
                    if ext == "json":
                        with open(f, "r", encoding="utf-8") as fh:
                            data = json.load(fh)
                        print(json.dumps(data, indent=2)[:500])
                    else:
                        with open(f, "r", encoding="utf-8", errors="ignore") as fh:
                            print(fh.read()[:500])
                except Exception as e:
                    print(f"  couldn't preview: {e}")

    def basic_stats(self) -> pd.DataFrame:
        rows = []
        for ext, files in self.file_index.items():
            for f in files:
                try:
                    size_kb = os.path.getsize(f) / 1024
                    rows.append({"file": os.path.basename(f), "ext": ext, "size_kb": size_kb})
                except Exception:
                    continue
        return pd.DataFrame(rows)

    def plot_file_sizes(self, stats_df: pd.DataFrame):
        if stats_df.empty:
            return
        fig, ax = plt.subplots(figsize=(10, 5))
        for ext in stats_df["ext"].unique():
            subset = stats_df[stats_df["ext"] == ext]
            ax.hist(subset["size_kb"], bins=20, alpha=0.6, label=ext)
        ax.set_xlabel("File size (KB)")
        ax.set_ylabel("Count")
        ax.set_title("Unrated Transcripts — File Size Distribution by Type")
        ax.legend()
        plt.tight_layout()
        savefig(fig, "06_unrated_transcripts_filesizes.pdf")
        plt.show()


# =====================================================================
# SECTION 5 — AMAZON MASSIVE (text NLU: intents + slots, not ASR)
# =====================================================================

class MassiveCorpus:

    def __init__(self, root: str):
        self.root = root
        self.df: Optional[pd.DataFrame] = None
        self.locale_used: Optional[str] = None

    def load(self, preferred_locale: str = "en-US"):
        if not os.path.isdir(self.root):
            log.warning("MASSIVE root missing, skipping.")
            return
        files = glob.glob(os.path.join(self.root, "**", "*.jsonl"), recursive=True)
        if not files:
            log.warning("No .jsonl files found under MASSIVE root.")
            return

        target = None
        for f in files:
            if preferred_locale in os.path.basename(f):
                target = f
                break
        target = target or files[0]
        self.locale_used = os.path.basename(target)

        records = []
        with open(target, "r", encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        self.df = pd.DataFrame(records)
        log.info(f"Loaded MASSIVE file: {target} -> {self.df.shape}")

    def plot_overview(self):
        if self.df is None:
            return
        fig = plt.figure(figsize=(20, 12))
        gs = gridspec.GridSpec(3, 3, figure=fig)

        # Intent distribution
        ax0 = fig.add_subplot(gs[0, 0])
        if "intent" in self.df.columns:
            self.df["intent"].value_counts().head(15).plot(kind="barh", ax=ax0, color="#4C72B0")
            ax0.invert_yaxis()
            ax0.set_title("Top 15 intents")

        # Utterance length
        ax1 = fig.add_subplot(gs[0, 1])
        if "utt" in self.df.columns:
            lens = self.df["utt"].astype(str).str.split().apply(len)
            ax1.hist(lens, bins=25, color="#55A868", edgecolor="white")
            ax1.set_title("Utterance length (words)")

        # Domain distribution
        ax2 = fig.add_subplot(gs[0, 2])
        if "domain" in self.df.columns:
            self.df["domain"].value_counts().plot(kind="bar", ax=ax2, color="#C44E52")
            ax2.set_title("Domain distribution")
        elif "scenario" in self.df.columns:
            self.df["scenario"].value_counts().plot(kind="bar", ax=ax2, color="#C44E52")
            ax2.set_title("Scenario distribution")

        # Slot count per utterance
        ax3 = fig.add_subplot(gs[1, 0])
        if "slots" in self.df.columns or "annot_utt" in self.df.columns:
            slot_col = "annot_utt" if "annot_utt" in self.df.columns else "slots"
            slot_tag_counts = self.df[slot_col].astype(str).str.count(r"\[")
            ax3.hist(slot_tag_counts, bins=range(0, slot_tag_counts.max() + 2), color="#8172B2")
            ax3.set_title("Slot tags per utterance")

        # Slot type distribution (if available)
        ax4 = fig.add_subplot(gs[1, 1])
        if "slot_types" in self.df.columns:
            slot_types = self.df["slot_types"].explode().value_counts().head(10)
            slot_types.plot(kind="bar", ax=ax4, color="#CCB974")
            ax4.set_title("Top slot types")

        # Pair plot for numeric features (if any)
        ax5 = fig.add_subplot(gs[1, 2])
        # Check for any numeric columns
        numeric_cols = self.df.select_dtypes(include=np.number).columns
        if len(numeric_cols) >= 2:
            # Scatter matrix - we'll just show two
            c1, c2 = numeric_cols[:2]
            ax5.scatter(self.df[c1], self.df[c2], alpha=0.3)
            ax5.set_xlabel(c1)
            ax5.set_ylabel(c2)
            ax5.set_title(f"{c1} vs {c2}")
        else:
            ax5.axis('off')
            ax5.text(0.5, 0.5, "No numeric columns", ha="center")

        # Intent vs Domain heatmap (if both)
        ax6 = fig.add_subplot(gs[2, 0])
        if "intent" in self.df.columns and "domain" in self.df.columns:
            cross = pd.crosstab(self.df["intent"], self.df["domain"])
            sns.heatmap(cross, ax=ax6, cmap="Blues", cbar_kws={'label': 'Count'})
            ax6.set_title("Intent vs Domain")
        else:
            ax6.axis('off')

        # Word cloud (not implemented to keep dependencies light)
        ax7 = fig.add_subplot(gs[2, 1])
        ax7.axis('off')
        ax7.text(0.5, 0.5, "Word cloud placeholder", ha="center")

        # Dataset statistics table
        ax8 = fig.add_subplot(gs[2, 2])
        ax8.axis('off')
        stats_text = f"Total utterances: {len(self.df)}\n"
        stats_text += f"Unique intents: {self.df['intent'].nunique() if 'intent' in self.df else 'N/A'}\n"
        stats_text += f"Unique domains: {self.df['domain'].nunique() if 'domain' in self.df else 'N/A'}\n"
        stats_text += f"Avg. utterance length: {self.df['utt'].str.split().str.len().mean() if 'utt' in self.df else 'N/A'}"
        ax8.text(0, 0.5, stats_text, fontsize=12, verticalalignment='center')

        fig.suptitle(f"MASSIVE Overview — {self.locale_used}", fontsize=16)
        plt.tight_layout()
        savefig(fig, "07_massive_overview.pdf")
        plt.show()


# =====================================================================
# SECTION 6 — CHiME-6 (transcriptions + floorplans)
# =====================================================================

class Chime6Corpus:

    def __init__(self, trans_root: str, floor_root: str):
        self.trans_root = trans_root
        self.floor_root = floor_root
        self.segments_df: Optional[pd.DataFrame] = None

    def load_transcriptions(self):
        if not os.path.isdir(self.trans_root):
            log.warning("CHiME-6 transcriptions root missing, skipping.")
            return
        files = glob.glob(os.path.join(self.trans_root, "*.json"))
        if not files:
            log.warning("No .json transcription files found.")
            return

        all_segments = []
        for f in files:
            try:
                with open(f, "r", encoding="utf-8") as fh:
                    data = json.load(fh)
                if isinstance(data, list):
                    for seg in data:
                        seg["_session"] = os.path.basename(f)
                    all_segments.extend(data)
            except Exception as e:
                log.error(f"Failed to parse {f}: {e}")

        self.segments_df = pd.DataFrame(all_segments)
        log.info(f"Loaded {len(self.segments_df)} segments from {len(files)} sessions.")

    def plot_speaker_activity(self):
        if self.segments_df is None or self.segments_df.empty:
            return
        df = self.segments_df

        fig = plt.figure(figsize=(20, 12))
        gs = gridspec.GridSpec(2, 3, figure=fig)

        # Speaker turns
        ax0 = fig.add_subplot(gs[0, 0])
        if "speaker" in df.columns:
            df["speaker"].value_counts().plot(kind="bar", ax=ax0, color="#4C72B0")
            ax0.set_title("Turns per speaker (all sessions)")

        # Utterance durations
        ax1 = fig.add_subplot(gs[0, 1])
        if {"start_time", "end_time"}.issubset(df.columns):
            def to_seconds(t):
                try:
                    parts = str(t).split(":")
                    return sum(float(p) * 60 ** i for i, p in enumerate(reversed(parts)))
                except Exception:
                    return np.nan

            df["duration"] = df["end_time"].apply(to_seconds) - df["start_time"].apply(to_seconds)
            ax1.hist(df["duration"].dropna(), bins=30, color="#55A868", edgecolor="white")
            ax1.set_title("Utterance duration distribution")
            ax1.set_xlabel("Seconds")

        # Session-wise segment count
        ax2 = fig.add_subplot(gs[0, 2])
        if "_session" in df.columns:
            df["_session"].value_counts().plot(kind="bar", ax=ax2, color="#C44E52")
            ax2.set_title("Segments per session")

        # Speaker overlap (if start/end and speaker available)
        ax3 = fig.add_subplot(gs[1, 0])
        if {"start_time", "end_time", "speaker"}.issubset(df.columns):
            # Compute overlap counts (simplified: just histogram of active speakers per time bin)
            # Not implementing fully - just placeholder
            ax3.text(0.5, 0.5, "Speaker overlap analysis\n(requires time-aligned data)", ha="center")
            ax3.axis('off')

        # Text length by speaker
        ax4 = fig.add_subplot(gs[1, 1])
        if "speaker" in df.columns and "text" in df.columns:
            df_group = df.groupby("speaker")["text"].apply(lambda x: x.str.len().mean())
            df_group.plot(kind="bar", ax=ax4, color="#8172B2")
            ax4.set_title("Avg. text length by speaker")
            ax4.set_ylabel("Characters")

        # Word count distribution
        ax5 = fig.add_subplot(gs[1, 2])
        if "text" in df.columns:
            word_counts = df["text"].astype(str).str.split().apply(len)
            ax5.hist(word_counts, bins=20, color="#CCB974", edgecolor="white")
            ax5.set_title("Word count per utterance")
            ax5.set_xlabel("Words")

        fig.suptitle("CHiME-6 Transcription Analysis", fontsize=15)
        plt.tight_layout()
        savefig(fig, "08_chime6_speaker_activity.pdf")
        plt.show()

    def list_floorplans(self):
        if not os.path.isdir(self.floor_root):
            log.warning("CHiME-6 floorplans root missing.")
            return []
        images = glob.glob(os.path.join(self.floor_root, "*"))
        log.info(f"Found {len(images)} floorplan files.")
        return images


# =====================================================================
# SECTION 7 — STATISTICAL ANALYSIS AND MACHINE LEARNING
# =====================================================================

class StatisticalAnalyzer:

    def __init__(self, features_df: pd.DataFrame):
        self.features_df = features_df

    def plot_feature_pairplot(self):
        if self.features_df.empty:
            return
        # Select a subset of features for pairplot
        cols = ["duration_s", "centroid_mean", "bandwidth_mean", "zcr_mean", "rms_mean", "f0_mean"]
        cols = [c for c in cols if c in self.features_df.columns]
        if len(cols) < 2:
            log.warning("Not enough features for pairplot.")
            return
        df_sub = self.features_df[cols].dropna()
        if df_sub.shape[0] < 5:
            return
        g = sns.pairplot(df_sub, diag_kind='kde', corner=True)
        g.fig.suptitle("Pairwise relationships among acoustic features", y=1.02)
        plt.tight_layout()
        savefig(g.fig, "10_pairplot_acoustic_features.pdf")
        plt.show()

    def perform_anova(self, group_col: str = None):
        if group_col is None or group_col not in self.features_df.columns:
            log.warning("No grouping column available for ANOVA.")
            return
        # Select numeric features
        numeric_cols = self.features_df.select_dtypes(include=np.number).columns
        numeric_cols = [c for c in numeric_cols if c not in ["duration_s"]]  # skip duration
        results = []
        for col in numeric_cols:
            groups = self.features_df.groupby(group_col)[col].apply(list)
            # Only if each group has >1 sample and at least 2 groups
            if len(groups) >= 2 and all(len(g) > 1 for g in groups):
                f_stat, p_val = stats.f_oneway(*groups)
                results.append({"feature": col, "F": f_stat, "p": p_val})
        if results:
            results_df = pd.DataFrame(results)
            results_df["p_adj"] = stats.false_discovery_control(results_df["p"])  # FDR correction
            results_df.to_csv(os.path.join(PATHS.output_dir, "anova_results.csv"), index=False)
            log.info("ANOVA results saved.")
            # Plot
            fig, ax = plt.subplots(figsize=(10, 6))
            sig = results_df[results_df["p_adj"] < 0.05]
            ax.barh(results_df["feature"], -np.log10(results_df["p_adj"]), color="skyblue")
            ax.axvline(-np.log10(0.05), color="red", linestyle="--", label="α=0.05")
            ax.set_xlabel("-log10(adjusted p-value)")
            ax.set_title(f"ANOVA significance across groups ({group_col})")
            ax.legend()
            plt.tight_layout()
            savefig(fig, "11_anova_results.pdf")
            plt.show()
        else:
            log.warning("No valid ANOVA tests performed.")


class MLModel:

    def __init__(self, features_df: pd.DataFrame):
        self.features_df = features_df
        self.X = None
        self.y = None
        self.model = None
        self.scaler = StandardScaler()

    def prepare_data(self, target_col: str = "gender"):
        if self.features_df.empty:
            return False
        if target_col not in self.features_df.columns:
            log.warning(f"Target column '{target_col}' not found. Using synthetic target.")
            # Create synthetic binary target based on f0_mean threshold
            if "f0_mean" in self.features_df.columns:
                median_f0 = self.features_df["f0_mean"].median()
                self.features_df["synthetic_gender"] = (self.features_df["f0_mean"] > median_f0).astype(int)
                target_col = "synthetic_gender"
                log.info(f"Created synthetic gender based on F0 median {median_f0:.1f} Hz")
            else:
                log.warning("No feature to generate synthetic target. Skipping ML.")
                return False

        # Select numeric features
        numeric_cols = self.features_df.select_dtypes(include=np.number).columns
        # Exclude target and any identifiers
        exclude = [target_col, "duration_s", "path"] if "path" in self.features_df.columns else [target_col]
        feature_cols = [c for c in numeric_cols if c not in exclude and not c.startswith("formants")]
        if len(feature_cols) == 0:
            log.warning("No numeric features available for ML.")
            return False

        self.X = self.features_df[feature_cols].dropna()
        self.y = self.features_df.loc[self.X.index, target_col].astype(int)
        self.X = self.scaler.fit_transform(self.X)
        log.info(f"Prepared data: X shape {self.X.shape}, y shape {self.y.shape}")
        return True

    def train_evaluate(self):
        if self.X is None or self.y is None:
            log.warning("Data not prepared. Call prepare_data() first.")
            return

        X_train, X_test, y_train, y_test = train_test_split(
            self.X, self.y, test_size=0.3, random_state=42, stratify=self.y
        )

        models = {
            "Random Forest": RandomForestClassifier(n_estimators=50, random_state=42),
            "SVM": SVC(kernel='rbf', C=1.0, gamma='scale'),
            "KNN": KNeighborsClassifier(n_neighbors=5),
        }

        results = []
        for name, clf in models.items():
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            acc = accuracy_score(y_test, y_pred)
            results.append({"model": name, "accuracy": acc})
            # Cross-validation
            cv_scores = cross_val_score(clf, self.X, self.y, cv=3)
            results[-1]["cv_mean"] = cv_scores.mean()
            results[-1]["cv_std"] = cv_scores.std()

        results_df = pd.DataFrame(results)
        log.info("Model performance:\n" + results_df.to_string())
        results_df.to_csv(os.path.join(PATHS.output_dir, "ml_results.csv"), index=False)

        # Plot confusion matrix for best model
        best_model_name = results_df.loc[results_df["accuracy"].idxmax(), "model"]
        best_clf = models[best_model_name]
        best_clf.fit(X_train, y_train)
        y_pred = best_clf.predict(X_test)
        cm = confusion_matrix(y_test, y_pred)

        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title(f"Confusion Matrix - {best_model_name}")
        plt.tight_layout()
        savefig(fig, "12_best_model_confusion.pdf")
        plt.show()

        # Feature importance (for Random Forest)
        if best_model_name == "Random Forest":
            importances = best_clf.feature_importances_
            feature_names = self.features_df.select_dtypes(include=np.number).columns
            feature_names = [c for c in feature_names if c not in ["gender", "duration_s", "path"]]
            # Ensure lengths match
            if len(importances) == len(feature_names):
                imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
                imp_df = imp_df.sort_values("importance", ascending=False).head(10)
                fig, ax = plt.subplots(figsize=(8, 5))
                ax.barh(imp_df["feature"], imp_df["importance"], color="#4C72B0")
                ax.set_xlabel("Importance")
                ax.set_title("Top 10 Feature Importances (Random Forest)")
                plt.tight_layout()
                savefig(fig, "13_feature_importances.pdf")
                plt.show()

        return results_df


# =====================================================================
# SECTION 8 — QUANTUM-INSPIRED FEATURE MAPPING (simulated)
# =====================================================================

class QuantumFeatureMapper:
    def __init__(self, n_components: int = 8):
        self.n_components = n_components
        self.pca = PCA(n_components=n_components)
        self.scaler = StandardScaler()

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        # Standardize
        X_scaled = self.scaler.fit_transform(X)
        # Apply PCA to reduce dimension
        X_pca = self.pca.fit_transform(X_scaled)
        # Create complex embedding: real part = PCA components, imaginary part = shifted version
        # Or use a random phase rotation
        np.random.seed(42)
        phase = np.exp(1j * np.random.uniform(0, 2*np.pi, (self.n_components, self.n_components)))
        # Complex projection
        X_complex = X_pca @ phase
        # Compute amplitude and phase features
        amplitude = np.abs(X_complex)
        phase_angle = np.angle(X_complex)
        # Concatenate real and imaginary parts (or amplitude/phase)
        X_quantum = np.concatenate([amplitude, phase_angle], axis=1)
        return X_quantum

    def plot_quantum_features(self, X_quantum: np.ndarray, labels: Optional[np.ndarray] = None):
        if X_quantum.shape[1] < 2:
            log.warning("Quantum features dimension <2, cannot plot.")
            return

        # Ensure no NaN or Inf values
        if not np.isfinite(X_quantum).all():
            log.warning("X_quantum contains NaN or Inf. Replacing with zeros.")
            X_quantum = np.nan_to_num(X_quantum, nan=0.0, posinf=0.0, neginf=0.0)

        # Try t-SNE; fall back to PCA if it fails
        try:
            # Use t-SNE for visualization with a safe perplexity
            n_samples = X_quantum.shape[0]
            perplexity = min(30, n_samples - 1)
            tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
            X_tsne = tsne.fit_transform(X_quantum)
            method = "t-SNE"
        except Exception as e:
            log.warning(f"t-SNE failed: {e}. Falling back to PCA.")
            from sklearn.decomposition import PCA
            pca = PCA(n_components=2)
            X_tsne = pca.fit_transform(X_quantum)
            method = "PCA"

        # Plot
        fig, ax = plt.subplots(figsize=(8, 6))
        if labels is not None:
            # Ensure labels are numeric and finite
            labels = np.asarray(labels)
            if labels.dtype.kind not in 'iuf':
                # Convert string labels to numeric categories
                unique_labels = np.unique(labels)
                label_map = {lbl: i for i, lbl in enumerate(unique_labels)}
                labels_num = np.array([label_map[lbl] for lbl in labels])
                scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels_num, cmap='viridis', alpha=0.7)
                ax.legend(*scatter.legend_elements(), title="Class")
            else:
                scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels, cmap='viridis', alpha=0.7)
                ax.legend(*scatter.legend_elements(), title="Class")
        else:
            ax.scatter(X_tsne[:, 0], X_tsne[:, 1], alpha=0.7)

        ax.set_title(f"Quantum-Inspired Features ({method})")
        ax.set_xlabel(f"{method} 1")
        ax.set_ylabel(f"{method} 2")
        plt.tight_layout()
        savefig(fig, "14_quantum_tsne.pdf")
        plt.show()

        # Also plot the quantum feature matrix heatmap (first 20 samples)
        if X_quantum.shape[0] > 20:
            fig, ax = plt.subplots(figsize=(10, 6))
            im = ax.imshow(X_quantum[:20, :], aspect='auto', cmap='RdBu_r')
            ax.set_title("Quantum Feature Matrix (first 20 samples)")
            ax.set_xlabel("Feature index")
            ax.set_ylabel("Sample index")
            plt.colorbar(im)
            plt.tight_layout()
            savefig(fig, "15_quantum_feature_matrix.pdf")
            plt.show()


# =====================================================================
# SECTION 9 — CROSS-DATASET COMPARISON AND REPORTING
# =====================================================================

def cross_dataset_summary(cv_features: pd.DataFrame, asr_results: pd.DataFrame):
    fig = plt.figure(figsize=(18, 8))
    gs = gridspec.GridSpec(2, 3, figure=fig)

    # Common Voice feature statistics
    ax0 = fig.add_subplot(gs[0, 0])
    if not cv_features.empty:
        numeric_cols = cv_features.select_dtypes(include=np.number).columns
        stats_df = cv_features[numeric_cols].describe().T
        # Display selected stats
        ax0.axis('off')
        ax0.table(cellText=stats_df[["mean", "std", "min", "max"]].round(2).values,
                  rowLabels=stats_df.index,
                  colLabels=["Mean", "Std", "Min", "Max"],
                  cellLoc='center', loc='center')
        ax0.set_title("Common Voice Features Summary")
    else:
        ax0.text(0.5, 0.5, "No CV audio features available", ha="center")
        ax0.axis("off")

    # ASR error summary
    ax1 = fig.add_subplot(gs[0, 1])
    if not asr_results.empty:
        ax1.boxplot([asr_results["wer"], asr_results["cer"]], labels=["WER", "CER"])
        ax1.set_title("ASR baseline error summary")
        ax1.set_ylabel("Error rate")
    else:
        ax1.text(0.5, 0.5, "No ASR results available", ha="center")
        ax1.axis("off")

    # Correlation between features and WER
    ax2 = fig.add_subplot(gs[0, 2])
    if not cv_features.empty and not asr_results.empty:
        # Merge by path
        merged = pd.merge(cv_features, asr_results, on="path", how="inner")
        if not merged.empty:
            numeric_cols = merged.select_dtypes(include=np.number).columns
            # Compute correlation with WER
            corrs = merged[numeric_cols].corr()["wer"].drop("wer").sort_values()
            top = corrs.head(5)
            bottom = corrs.tail(5)
            combined = pd.concat([top, bottom])
            ax2.barh(combined.index, combined.values, color=["red" if v<0 else "blue" for v in combined.values])
            ax2.set_xlabel("Correlation with WER")
            ax2.set_title("Features correlated with WER")
        else:
            ax2.text(0.5, 0.5, "No overlap between CV and ASR", ha="center")
            ax2.axis("off")
    else:
        ax2.text(0.5, 0.5, "Insufficient data", ha="center")
        ax2.axis("off")

    # Dataset size comparison
    ax3 = fig.add_subplot(gs[1, 0])
    sizes = {}
    if cv_features is not None:
        sizes["Common Voice clips"] = len(cv_features)
    if asr_results is not None:
        sizes["ASR evaluated"] = len(asr_results)
    # Add MASSIVE size if available
    if STATUS.get("massive_root", False):
        try:
            massive = MassiveCorpus(PATHS.massive_root)
            massive.load()
            if massive.df is not None:
                sizes["MASSIVE utterances"] = len(massive.df)
        except:
            pass
    if sizes:
        ax3.bar(sizes.keys(), sizes.values(), color="#4C72B0")
        ax3.set_ylabel("Count")
        ax3.set_title("Dataset sizes")
    else:
        ax3.axis('off')
        ax3.text(0.5, 0.5, "No dataset sizes", ha="center")

    # Sentence length comparison across datasets
    ax4 = fig.add_subplot(gs[1, 1])
    # We can plot length distributions from CV and MASSIVE if available
    cv_len = None
    massive_len = None
    # CV
    cv = CommonVoiceCorpus(PATHS.cv_root)
    cv.load_tsvs()
    validated = cv.get("validated")
    if validated is not None:
        cv_len = validated["sentence"].astype(str).str.len()
    # MASSIVE
    massive = MassiveCorpus(PATHS.massive_root)
    massive.load()
    if massive.df is not None and "utt" in massive.df.columns:
        massive_len = massive.df["utt"].astype(str).str.len()
    if cv_len is not None or massive_len is not None:
        ax4.hist(cv_len, bins=30, alpha=0.5, label="Common Voice", color="blue")
        ax4.hist(massive_len, bins=30, alpha=0.5, label="MASSIVE", color="green")
        ax4.set_xlabel("Sentence length (chars)")
        ax4.set_ylabel("Count")
        ax4.legend()
        ax4.set_title("Sentence length distribution")
    else:
        ax4.axis('off')
        ax4.text(0.5, 0.5, "No length data", ha="center")

    # Quantum-inspired feature map placeholder
    ax5 = fig.add_subplot(gs[1, 2])
    ax5.text(0.5, 0.5, "Quantum feature analysis\n(see separate plots)", ha="center")
    ax5.axis('off')

    fig.suptitle("Cross-Dataset Summary", fontsize=16)
    plt.tight_layout()
    savefig(fig, "09_cross_dataset_summary.pdf")
    plt.show()


# =====================================================================
# SECTION 10 — ADVANCED VISUALIZATION: MULTI-PANEL SUBPLOTS
# =====================================================================

class AdvancedVisualizer:


    @staticmethod
    def plot_all_acoustic_aggregates(features_df: pd.DataFrame):
        if features_df.empty:
            return

        fig = plt.figure(figsize=(20, 16))
        gs = gridspec.GridSpec(4, 4, figure=fig, hspace=0.4, wspace=0.3)

        # 1. Duration distribution
        ax = fig.add_subplot(gs[0, 0])
        ax.hist(features_df["duration_s"], bins=30, color="#4C72B0", edgecolor="white")
        ax.set_title("Duration (s)")

        # 2. F0 distribution
        ax = fig.add_subplot(gs[0, 1])
        ax.hist(features_df["f0_mean"].dropna(), bins=30, color="#55A868", edgecolor="white")
        ax.set_title("Mean F0 (Hz)")

        # 3. Spectral centroid
        ax = fig.add_subplot(gs[0, 2])
        ax.hist(features_df["centroid_mean"].dropna(), bins=30, color="#C44E52", edgecolor="white")
        ax.set_title("Spectral centroid (Hz)")

        # 4. ZCR
        ax = fig.add_subplot(gs[0, 3])
        ax.hist(features_df["zcr_mean"].dropna(), bins=30, color="#8172B2", edgecolor="white")
        ax.set_title("Zero-crossing rate")

        # 5. RMS energy
        ax = fig.add_subplot(gs[1, 0])
        ax.hist(features_df["rms_mean"].dropna(), bins=30, color="#CCB974", edgecolor="white")
        ax.set_title("RMS energy")

        # 6. Spectral flatness
        if "flatness_mean" in features_df.columns:
            ax = fig.add_subplot(gs[1, 1])
            ax.hist(features_df["flatness_mean"].dropna(), bins=30, color="#9467BD", edgecolor="white")
            ax.set_title("Spectral flatness")

        # 7. Voiced fraction
        if "voiced_fraction" in features_df.columns:
            ax = fig.add_subplot(gs[1, 2])
            ax.hist(features_df["voiced_fraction"].dropna(), bins=30, color="#8C564B", edgecolor="white")
            ax.set_title("Voiced fraction")

        # 8. Speaking rate
        if "speaking_rate" in features_df.columns:
            ax = fig.add_subplot(gs[1, 3])
            ax.hist(features_df["speaking_rate"].dropna(), bins=30, color="#E377C2", edgecolor="white")
            ax.set_title("Speaking rate (peaks/s)")

        # 9-12: Scatter plots showing relationships
        ax = fig.add_subplot(gs[2, 0])
        ax.scatter(features_df["f0_mean"], features_df["centroid_mean"], alpha=0.5)
        ax.set_xlabel("F0 (Hz)")
        ax.set_ylabel("Centroid (Hz)")
        ax.set_title("F0 vs Centroid")

        ax = fig.add_subplot(gs[2, 1])
        ax.scatter(features_df["duration_s"], features_df["rms_mean"], alpha=0.5)
        ax.set_xlabel("Duration (s)")
        ax.set_ylabel("RMS")
        ax.set_title("Duration vs RMS")

        ax = fig.add_subplot(gs[2, 2])
        ax.scatter(features_df["zcr_mean"], features_df["centroid_mean"], alpha=0.5)
        ax.set_xlabel("ZCR")
        ax.set_ylabel("Centroid (Hz)")
        ax.set_title("ZCR vs Centroid")

        ax = fig.add_subplot(gs[2, 3])
        ax.scatter(features_df["voiced_fraction"], features_df["f0_mean"], alpha=0.5)
        ax.set_xlabel("Voiced fraction")
        ax.set_ylabel("F0 (Hz)")
        ax.set_title("Voiced fraction vs F0")

        # 13-16: Boxplots of features by some category (if available)
        # Use gender if available
        if "gender" in features_df.columns:
            gb = features_df.groupby("gender")
            ax = fig.add_subplot(gs[3, 0])
            data = [gb.get_group(g)["f0_mean"].dropna() for g in gb.groups if len(gb.get_group(g)) > 1]
            if data:
                ax.boxplot(data, labels=[str(g) for g in gb.groups if len(gb.get_group(g)) > 1])
                ax.set_title("F0 by gender")
            ax = fig.add_subplot(gs[3, 1])
            data = [gb.get_group(g)["centroid_mean"].dropna() for g in gb.groups if len(gb.get_group(g)) > 1]
            if data:
                ax.boxplot(data, labels=[str(g) for g in gb.groups if len(gb.get_group(g)) > 1])
                ax.set_title("Centroid by gender")
            ax = fig.add_subplot(gs[3, 2])
            data = [gb.get_group(g)["rms_mean"].dropna() for g in gb.groups if len(gb.get_group(g)) > 1]
            if data:
                ax.boxplot(data, labels=[str(g) for g in gb.groups if len(gb.get_group(g)) > 1])
                ax.set_title("RMS by gender")
            ax = fig.add_subplot(gs[3, 3])
            data = [gb.get_group(g)["zcr_mean"].dropna() for g in gb.groups if len(gb.get_group(g)) > 1]
            if data:
                ax.boxplot(data, labels=[str(g) for g in gb.groups if len(gb.get_group(g)) > 1])
                ax.set_title("ZCR by gender")

        fig.suptitle("Comprehensive Acoustic Feature Analysis", fontsize=18)
        plt.tight_layout()
        savefig(fig, "16_comprehensive_acoustic_analysis.pdf")
        plt.show()


# =====================================================================
# SECTION 11 — MAIN PIPELINE
# =====================================================================

def main():
    log.info("=== Starting pipeline ===")

    # --- Common Voice ---
    cv = CommonVoiceCorpus(PATHS.cv_root)
    cv.load_tsvs()
    print(cv.summarize())
    cv.plot_metadata_overview()

    sample_paths = cv.sample_clip_paths(n=50)  # increased sample size

    # Advanced audio feature extraction
    extractor = AdvancedAudioFeatureExtractor()
    cv_features = pd.DataFrame()
    if sample_paths:
        extractor.plot_single_clip(sample_paths[0])
        cv_features = extractor.batch_extract(sample_paths)
        extractor.plot_feature_distributions(cv_features)
        extractor.plot_feature_correlations(cv_features)
        cv_features.to_csv(os.path.join(PATHS.output_dir, "cv_audio_features.csv"), index=False)

    # --- ASR baseline ---
    asr_results = pd.DataFrame()
    if sample_paths:
        asr = ASRBaseline(model_size="small", language="as")
        asr_results = asr.evaluate(cv, sample_paths)
        asr.plot_results(asr_results)
        if not asr_results.empty:
            asr_results.to_csv(os.path.join(PATHS.output_dir, "asr_results.csv"), index=False)

    # --- Unrated transcripts ---
    unrated = UnratedTranscripts(PATHS.unrated_root)
    unrated.scan()
    unrated.preview(n=2)
    stats_df = unrated.basic_stats()
    unrated.plot_file_sizes(stats_df)

    # --- MASSIVE ---
    massive = MassiveCorpus(PATHS.massive_root)
    massive.load()
    massive.plot_overview()

    # --- CHiME-6 ---
    chime = Chime6Corpus(PATHS.chime_trans, PATHS.chime_floor)
    chime.load_transcriptions()
    chime.plot_speaker_activity()
    chime.list_floorplans()

    # --- Statistical Analysis ---
    if not cv_features.empty:
        stats_analyzer = StatisticalAnalyzer(cv_features)
        stats_analyzer.plot_feature_pairplot()
        if "gender" in cv_features.columns:
            stats_analyzer.perform_anova(group_col="gender")
        else:
            log.info("No gender column; skipping ANOVA.")

    # --- Machine Learning ---
    if not cv_features.empty:
        ml = MLModel(cv_features)
        if ml.prepare_data(target_col="gender"):  # tries gender, else synthetic
            ml.train_evaluate()
        else:
            log.warning("ML preparation failed.")

    # --- Quantum-inspired feature mapping ---
    if not cv_features.empty:
        numeric_cols = cv_features.select_dtypes(include=np.number).columns
        numeric_cols = [c for c in numeric_cols if c not in ["duration_s", "path"]]
        if len(numeric_cols) >= 2:
            X = cv_features[numeric_cols].dropna().values
            if X.shape[0] > 10 and X.shape[1] > 1:
                qmapper = QuantumFeatureMapper(n_components=min(8, X.shape[1]))
                X_q = qmapper.fit_transform(X)
                # Use synthetic labels for coloring if no gender
                labels = None
                if "gender" in cv_features.columns:
                    labels = cv_features.loc[cv_features[numeric_cols].dropna().index, "gender"].values
                qmapper.plot_quantum_features(X_q, labels)
                # Save quantum features
                np.save(os.path.join(PATHS.output_dir, "quantum_features.npy"), X_q)
                log.info("Quantum features saved.")

    # --- Advanced visualization ---
    if not cv_features.empty:
        AdvancedVisualizer.plot_all_acoustic_aggregates(cv_features)

    # --- Cross-dataset summary ---
    cross_dataset_summary(cv_features, asr_results)

    log.info("=== Pipeline complete. Check the pipeline_outputs/ folder. ===")


if __name__ == "__main__":
    main()

In [ ]:
# =====================================================================
# SECTION 12 — ADVANCED EXPERIMENTAL ANALYSIS
# =====================================================================

import textwrap
from collections import defaultdict
import re

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False

try:
    from wordcloud import WordCloud
    WORDCLOUD_AVAILABLE = True
except ImportError:
    WORDCLOUD_AVAILABLE = False

try:
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
    from sklearn.decomposition import LatentDirichletAllocation
    LDA_AVAILABLE = True
except ImportError:
    LDA_AVAILABLE = False

try:
    from sklearn.model_selection import GridSearchCV, learning_curve
    GRID_AVAILABLE = True
except ImportError:
    GRID_AVAILABLE = False

try:
    from sklearn.metrics import roc_curve, auc, precision_recall_curve
    ROC_AVAILABLE = True
except ImportError:
    ROC_AVAILABLE = False


class ExperimentalAnalysis:

    def __init__(self, cv_features: pd.DataFrame,
                 asr_results: pd.DataFrame,
                 massive_df: Optional[pd.DataFrame],
                 chime_df: Optional[pd.DataFrame]):
        self.cv_features = cv_features
        self.asr_results = asr_results
        self.massive_df = massive_df
        self.chime_df = chime_df

    def run_all(self):
        self.plot_asr_error_distribution()
        self.plot_wer_vs_acoustic_features()
        self.plot_correlation_heatmap_with_wer()
        self.plot_error_type_breakdown()
        self.plot_learning_curves_ml()
        self.plot_confusion_matrix_intent()
        self.plot_speaker_clustering()
        self.plot_utterance_length_effects()
        self.plot_word_clouds()
        self.plot_topic_modeling()
        self.plot_roc_curves()
        self.plot_feature_importance_shap()
        self.plot_code_switch_analysis()  # simulated if no real data
        self.plot_duration_vs_accuracy()
        self.plot_session_activity_chime()
        self.generate_latex_tables()

    # -----------------------------------------------------------------
    # 1. ASR Error Distribution
    # -----------------------------------------------------------------
    def plot_asr_error_distribution(self):
        if self.asr_results.empty:
            log.warning("Skipping ASR error distribution: no results.")
            return

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        ax = axes[0, 0]
        ax.hist(self.asr_results['wer'], bins=20, color='#1f77b4', edgecolor='white', alpha=0.7)
        ax.axvline(self.asr_results['wer'].mean(), color='red', linestyle='--', label=f'mean={self.asr_results["wer"].mean():.2f}')
        ax.axvline(self.asr_results['wer'].median(), color='green', linestyle='--', label=f'median={self.asr_results["wer"].median():.2f}')
        ax.set_xlabel('WER')
        ax.set_ylabel('Count')
        ax.set_title('WER Distribution')
        ax.legend()

        ax = axes[0, 1]
        ax.hist(self.asr_results['cer'], bins=20, color='#ff7f0e', edgecolor='white', alpha=0.7)
        ax.axvline(self.asr_results['cer'].mean(), color='red', linestyle='--', label=f'mean={self.asr_results["cer"].mean():.2f}')
        ax.axvline(self.asr_results['cer'].median(), color='green', linestyle='--', label=f'median={self.asr_results["cer"].median():.2f}')
        ax.set_xlabel('CER')
        ax.set_ylabel('Count')
        ax.set_title('CER Distribution')
        ax.legend()

        ax = axes[1, 0]
        # Scatter WER vs CER
        ax.scatter(self.asr_results['wer'], self.asr_results['cer'], alpha=0.6)
        ax.set_xlabel('WER')
        ax.set_ylabel('CER')
        ax.set_title('WER vs CER')
        # Add diagonal line
        lims = [0, max(self.asr_results['wer'].max(), self.asr_results['cer'].max())]
        ax.plot(lims, lims, 'k--', alpha=0.5, label='y=x')
        ax.legend()

        ax = axes[1, 1]
        # Cumulative distribution
        sorted_wer = np.sort(self.asr_results['wer'])
        cdf = np.arange(1, len(sorted_wer)+1) / len(sorted_wer)
        ax.plot(sorted_wer, cdf, marker='.', linestyle='-', color='#2ca02c')
        ax.set_xlabel('WER')
        ax.set_ylabel('Cumulative Probability')
        ax.set_title('CDF of WER')
        ax.grid(True, linestyle='--', alpha=0.5)

        fig.suptitle('ASR Error Analysis', fontsize=16)
        plt.tight_layout()
        savefig(fig, '20_asr_error_distribution.pdf')
        plt.show()

    # -----------------------------------------------------------------
    # 2. WER vs Acoustic Features (scatter + regression)
    # -----------------------------------------------------------------
    def plot_wer_vs_acoustic_features(self):
        if self.cv_features.empty or self.asr_results.empty:
            log.warning("Skipping WER vs features: missing data.")
            return

        # Merge on path
        merged = pd.merge(self.cv_features, self.asr_results, on='path', how='inner')
        if merged.empty:
            return

        features = ['duration_s', 'f0_mean', 'centroid_mean', 'bandwidth_mean',
                    'zcr_mean', 'rms_mean', 'voiced_fraction', 'speaking_rate']
        features = [f for f in features if f in merged.columns]

        n = len(features)
        cols = 4
        rows = (n + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(20, 5*rows))
        axes = axes.flatten() if rows > 1 else [axes]

        for i, feat in enumerate(features):
            ax = axes[i]
            ax.scatter(merged[feat], merged['wer'], alpha=0.6, s=30)
            # Linear regression
            mask = ~np.isnan(merged[feat]) & ~np.isnan(merged['wer'])
            if mask.sum() > 3:
                slope, intercept = np.polyfit(merged.loc[mask, feat], merged.loc[mask, 'wer'], 1)
                x_vals = np.linspace(merged[feat].min(), merged[feat].max(), 100)
                ax.plot(x_vals, slope*x_vals + intercept, 'r--', label=f'slope={slope:.3f}')
                ax.legend()
            ax.set_xlabel(feat)
            ax.set_ylabel('WER')
            ax.set_title(f'WER vs {feat}')
            ax.grid(True, linestyle='--', alpha=0.3)

        for j in range(len(features), len(axes)):
            axes[j].axis('off')

        fig.suptitle('Relationship between Acoustic Features and WER', fontsize=16)
        plt.tight_layout()
        savefig(fig, '21_wer_vs_features.pdf')
        plt.show()

    # -----------------------------------------------------------------
    # 3. Correlation Heatmap with WER
    # -----------------------------------------------------------------
    def plot_correlation_heatmap_with_wer(self):
        if self.cv_features.empty or self.asr_results.empty:
            return
        merged = pd.merge(self.cv_features, self.asr_results, on='path', how='inner')
        if merged.empty:
            return

        # Select numeric columns
        numeric_cols = merged.select_dtypes(include=np.number).columns
        # Exclude identifier columns
        exclude = ['path', 'ref_len_chars', 'ref_len_words']
        numeric_cols = [c for c in numeric_cols if c not in exclude]
        if 'wer' not in numeric_cols:
            return
        corr = merged[numeric_cols].corr()
        # Sort by correlation with WER
        wer_corr = corr['wer'].drop('wer').sort_values()
        # Take top 15 features
        top_features = wer_corr.abs().nlargest(15).index
        corr_sub = corr.loc[top_features, top_features]

        fig, ax = plt.subplots(figsize=(12, 10))
        mask = np.triu(np.ones_like(corr_sub, dtype=bool))
        sns.heatmap(corr_sub, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                    center=0, square=True, linewidths=0.5, ax=ax,
                    cbar_kws={'label': 'Correlation'})
        ax.set_title('Correlation Matrix of Top Features (with WER)')
        plt.tight_layout()
        savefig(fig, '22_correlation_with_wer.pdf')
        plt.show()

    # -----------------------------------------------------------------
    # 4. Error Type Breakdown (insertions, deletions, substitutions)
    # -----------------------------------------------------------------
    def plot_error_type_breakdown(self):
        if self.asr_results.empty:
            return
        try:
            from jiwer import compute_measures
            ins, dels, subs = [], [], []
            for _, row in self.asr_results.iterrows():
                m = compute_measures(row['reference'], row['hypothesis'])
                ins.append(m['insertions'])
                dels.append(m['deletions'])
                subs.append(m['substitutions'])
            df_err = pd.DataFrame({'Insertions': ins, 'Deletions': dels, 'Substitutions': subs})

            fig, axes = plt.subplots(1, 2, figsize=(14, 6))

            ax = axes[0]
            df_err.mean().plot(kind='bar', ax=ax, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
            ax.set_ylabel('Mean count per utterance')
            ax.set_title('Average Error Type Counts')
            ax.grid(True, axis='y', linestyle='--', alpha=0.5)

            ax = axes[1]
            # Stacked histogram
            df_err.sum(axis=1).hist(bins=20, color='gray', alpha=0.7, edgecolor='white')
            ax.set_xlabel('Total errors per utterance')
            ax.set_ylabel('Count')
            ax.set_title('Total Error Count Distribution')

            fig.suptitle('Error Type Breakdown', fontsize=16)
            plt.tight_layout()
            savefig(fig, '23_error_type_breakdown.pdf')
            plt.show()
        except Exception as e:
            log.warning(f"Error type breakdown failed: {e}")

    # -----------------------------------------------------------------
    # 5. Learning Curves for ML Models
    # -----------------------------------------------------------------
    def plot_learning_curves_ml(self):
        if self.cv_features.empty:
            return
        # Prepare data for classification (using synthetic target if needed)
        ml = MLModel(self.cv_features)
        if not ml.prepare_data(target_col='gender'):
            log.warning("Skipping learning curves: no target.")
            return

        X, y = ml.X, ml.y
        if X is None or y is None:
            return

        from sklearn.model_selection import learning_curve
        from sklearn.ensemble import RandomForestClassifier

        model = RandomForestClassifier(n_estimators=30, random_state=42)
        train_sizes, train_scores, test_scores = learning_curve(
            model, X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 10),
            scoring='accuracy', n_jobs=-1
        )
        train_mean = np.mean(train_scores, axis=1)
        train_std = np.std(train_scores, axis=1)
        test_mean = np.mean(test_scores, axis=1)
        test_std = np.std(test_scores, axis=1)

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2, color='blue')
        ax.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.2, color='green')
        ax.plot(train_sizes, train_mean, 'o-', color='blue', label='Training accuracy')
        ax.plot(train_sizes, test_mean, 'o-', color='green', label='Validation accuracy')
        ax.set_xlabel('Training examples')
        ax.set_ylabel('Accuracy')
        ax.set_title('Learning Curves (Random Forest)')
        ax.legend(loc='best')
        ax.grid(True, linestyle='--', alpha=0.3)
        plt.tight_layout()
        savefig(fig, '24_learning_curves.pdf')
        plt.show()

    # -----------------------------------------------------------------
    # 6. Confusion Matrix for Intent Classification (MASSIVE)
    # -----------------------------------------------------------------
    def plot_confusion_matrix_intent(self):
        if self.massive_df is None or self.massive_df.empty:
            log.warning("MASSIVE data not available; skipping intent confusion.")
            return

        df = self.massive_df
        if 'intent' not in df.columns or 'utt' not in df.columns:
            log.warning("MASSIVE lacks 'intent' or 'utt' columns.")
            return

        # Use a simple text classifier to show confusion
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.svm import LinearSVC
        from sklearn.model_selection import train_test_split

        # Take a subset to speed up
        df_sub = df.sample(min(10000, len(df)), random_state=42)
        X_text = df_sub['utt'].astype(str)
        y_intent = df_sub['intent']

        # Only keep top intents with at least 30 samples
        top_intents = y_intent.value_counts()[y_intent.value_counts() >= 30].index
        mask = y_intent.isin(top_intents)
        X_text = X_text[mask]
        y_intent = y_intent[mask]

        if len(np.unique(y_intent)) < 2:
            return

        vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
        X_vec = vectorizer.fit_transform(X_text)

        X_train, X_test, y_train, y_test = train_test_split(
            X_vec, y_intent, test_size=0.3, random_state=42, stratify=y_intent
        )

        clf = LinearSVC(max_iter=1000, random_state=42)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
        # Normalize
        cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

        fig, ax = plt.subplots(figsize=(12, 10))
        sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                    xticklabels=clf.classes_, yticklabels=clf.classes_, ax=ax)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.set_title('Intent Classification Confusion Matrix (MASSIVE)')
        plt.tight_layout()
        savefig(fig, '25_intent_confusion.pdf')
        plt.show()

    # -----------------------------------------------------------------
    # 7. Speaker Clustering based on Acoustic Features
    # -----------------------------------------------------------------
    def plot_speaker_clustering(self):
        if self.cv_features.empty or 'client_id' not in self.cv_features.columns:
            log.warning("Skipping speaker clustering: no client_id in features.")
            return

        # Compute speaker-level feature averages
        feature_cols = ['f0_mean', 'centroid_mean', 'bandwidth_mean', 'zcr_mean',
                        'rms_mean', 'voiced_fraction', 'speaking_rate']
        feature_cols = [c for c in feature_cols if c in self.cv_features.columns]
        if len(feature_cols) < 2:
            return

        speaker_features = self.cv_features.groupby('client_id')[feature_cols].mean().dropna()
        if speaker_features.shape[0] < 3:
            return

        # Standardize
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(speaker_features)

        # Hierarchical clustering
        linkage_matrix = linkage(X_scaled, method='ward')

        fig, ax = plt.subplots(figsize=(12, 8))
        dendrogram(linkage_matrix, labels=speaker_features.index, ax=ax,
                   leaf_rotation=90, leaf_font_size=8)
        ax.set_title('Speaker Clustering (Acoustic Features)')
        ax.set_xlabel('Speaker ID')
        ax.set_ylabel('Distance')
        plt.tight_layout()
        savefig(fig, '26_speaker_clustering.pdf')
        plt.show()

        # Also PCA 2D projection
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_scaled)
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.7)
        # Label some points
        for i, idx in enumerate(speaker_features.index[:10]):
            ax.annotate(idx, (X_pca[i, 0], X_pca[i, 1]), fontsize=8)
        ax.set_xlabel('PC1')
        ax.set_ylabel('PC2')
        ax.set_title('PCA of Speakers by Acoustic Features')
        plt.tight_layout()
        savefig(fig, '27_speaker_pca.pdf')
        plt.show()

    # -----------------------------------------------------------------
    # 8. Utterance Length Effects (WER vs length)
    # -----------------------------------------------------------------
    def plot_utterance_length_effects(self):
        if self.asr_results.empty:
            return

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        ax = axes[0]
        ax.scatter(self.asr_results['ref_len_words'], self.asr_results['wer'], alpha=0.6)
        ax.set_xlabel('Reference length (words)')
        ax.set_ylabel('WER')
        ax.set_title('WER vs Utterance Length')
        # Add regression line
        mask = ~np.isnan(self.asr_results['ref_len_words']) & ~np.isnan(self.asr_results['wer'])
        if mask.sum() > 3:
            slope, intercept = np.polyfit(self.asr_results.loc[mask, 'ref_len_words'],
                                          self.asr_results.loc[mask, 'wer'], 1)
            x_vals = np.linspace(self.asr_results['ref_len_words'].min(),
                                 self.asr_results['ref_len_words'].max(), 100)
            ax.plot(x_vals, slope*x_vals + intercept, 'r--', label=f'slope={slope:.3f}')
            ax.legend()

        ax = axes[1]
        # Boxplot by length bin
        bins = [0, 5, 10, 15, 20, 25, 30, 100]
        labels = ['0-5', '6-10', '11-15', '16-20', '21-25', '26-30', '30+']
        self.asr_results['length_bin'] = pd.cut(self.asr_results['ref_len_words'],
                                                bins=bins, labels=labels, right=False)
        self.asr_results.boxplot(column='wer', by='length_bin', ax=ax)
        ax.set_title('WER by Utterance Length Bin')
        ax.set_xlabel('Length bin (words)')
        ax.set_ylabel('WER')
        ax.grid(True, linestyle='--', alpha=0.3)

        fig.suptitle('Effect of Utterance Length on ASR Performance', fontsize=16)
        plt.tight_layout()
        savefig(fig, '28_length_effect.pdf')
        plt.show()

    # -----------------------------------------------------------------
    # 9. Word Clouds from Transcripts
    # -----------------------------------------------------------------
    def plot_word_clouds(self):
        if not WORDCLOUD_AVAILABLE:
            log.warning("wordcloud not installed; skipping word clouds.")
            return

        # For Common Voice
        if not self.cv_features.empty and 'sentence' in self.cv_features.columns:
            text_cv = ' '.join(self.cv_features['sentence'].astype(str))
            wc = WordCloud(width=800, height=400, background_color='white').generate(text_cv)
            fig, ax = plt.subplots(figsize=(12, 6))
            ax.imshow(wc, interpolation='bilinear')
            ax.axis('off')
            ax.set_title('Common Voice - Word Cloud')
            plt.tight_layout()
            savefig(fig, '29_wordcloud_cv.pdf')
            plt.show()

        # For MASSIVE
        if self.massive_df is not None and not self.massive_df.empty and 'utt' in self.massive_df.columns:
            text_massive = ' '.join(self.massive_df['utt'].astype(str))
            wc = WordCloud(width=800, height=400, background_color='white').generate(text_massive)
            fig, ax = plt.subplots(figsize=(12, 6))
            ax.imshow(wc, interpolation='bilinear')
            ax.axis('off')
            ax.set_title('MASSIVE - Word Cloud')
            plt.tight_layout()
            savefig(fig, '30_wordcloud_massive.pdf')
            plt.show()

    # -----------------------------------------------------------------
    # 10. Topic Modeling (LDA) on Transcripts
    # -----------------------------------------------------------------
    def plot_topic_modeling(self):
        if not LDA_AVAILABLE:
            log.warning("LDA dependencies missing; skipping topic modeling.")
            return
        if self.massive_df is None or self.massive_df.empty:
            log.warning("MASSIVE data missing; skipping LDA.")
            return

        # Use MASSIVE utterances
        docs = self.massive_df['utt'].astype(str).values
        vectorizer = CountVectorizer(max_features=1000, stop_words='english',
                                     max_df=0.8, min_df=5)
        doc_term = vectorizer.fit_transform(docs)
        if doc_term.shape[0] < 10:
            return

        lda = LatentDirichletAllocation(n_components=5, random_state=42, max_iter=10)
        lda.fit(doc_term)

        # Display top words per topic
        feature_names = vectorizer.get_feature_names_out()
        n_top_words = 10
        topics = []
        for topic_idx, topic in enumerate(lda.components_):
            top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words-1:-1]]
            topics.append(f"Topic {topic_idx+1}: " + ", ".join(top_words))

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.axis('off')
        ax.text(0, 0.5, '\n'.join(topics), fontsize=12, verticalalignment='center')
        ax.set_title('LDA Topic Modeling on MASSIVE Utterances')
        plt.tight_layout()
        savefig(fig, '31_lda_topics.pdf')
        plt.show()

    # -----------------------------------------------------------------
    # 11. ROC Curves for ML Models (if binary classification)
    # -----------------------------------------------------------------
    def plot_roc_curves(self):
        if not ROC_AVAILABLE:
            return
        if self.cv_features.empty:
            return
        ml = MLModel(self.cv_features)
        if not ml.prepare_data(target_col='gender'):
            return
        X, y = ml.X, ml.y
        if X is None or y is None:
            return
        if len(np.unique(y)) != 2:
            log.warning("ROC requires binary target.")
            return

        from sklearn.ensemble import RandomForestClassifier
        from sklearn.linear_model import LogisticRegression
        from sklearn.svm import SVC
        from sklearn.model_selection import train_test_split
        from sklearn.metrics import roc_curve, auc

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=42, stratify=y
        )

        models = {
            'RF': RandomForestClassifier(n_estimators=30, random_state=42),
            'LR': LogisticRegression(max_iter=1000, random_state=42),
            'SVM': SVC(probability=True, random_state=42)
        }

        fig, ax = plt.subplots(figsize=(8, 6))
        for name, clf in models.items():
            clf.fit(X_train, y_train)
            if hasattr(clf, "predict_proba"):
                y_pred_proba = clf.predict_proba(X_test)[:, 1]
            else:
                continue
            fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
            auc_score = auc(fpr, tpr)
            ax.plot(fpr, tpr, label=f'{name} (AUC={auc_score:.3f})')

        ax.plot([0, 1], [0, 1], 'k--', label='Random')
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title('ROC Curves for Acoustic Classification')
        ax.legend(loc='lower right')
        plt.tight_layout()
        savefig(fig, '32_roc_curves.pdf')
        plt.show()

    # -----------------------------------------------------------------
    # 12. SHAP Feature Importance
    # -----------------------------------------------------------------
    def plot_feature_importance_shap(self):
        if not SHAP_AVAILABLE:
            log.warning("SHAP not installed; skipping SHAP analysis.")
            return
        if self.cv_features.empty:
            return
        ml = MLModel(self.cv_features)
        if not ml.prepare_data(target_col='gender'):
            return
        X, y = ml.X, ml.y
        if X is None or y is None:
            return

        from sklearn.ensemble import RandomForestClassifier
        model = RandomForestClassifier(n_estimators=30, random_state=42)
        model.fit(X, y)

        # Use a subset for SHAP (due to speed)
        X_sub = X[:100] if X.shape[0] > 100 else X
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_sub)

        # Summary plot
        fig, ax = plt.subplots(figsize=(10, 6))
        shap.summary_plot(shap_values[1] if isinstance(shap_values, list) else shap_values,
                          X_sub, feature_names=ml.features_df.columns, show=False)
        plt.tight_layout()
        savefig(fig, '33_shap_summary.pdf')
        plt.show()

        # Bar plot
        shap.summary_plot(shap_values[1] if isinstance(shap_values, list) else shap_values,
                          X_sub, feature_names=ml.features_df.columns,
                          plot_type="bar", show=False)
        plt.tight_layout()
        savefig(fig, '34_shap_bar.pdf')
        plt.show()

    # -----------------------------------------------------------------
    # 13. Code-Switch Analysis (simulated if no real data)
    # -----------------------------------------------------------------
    def plot_code_switch_analysis(self):
        # If we had code-switch data, we could plot it. Simulate.
        # Create a dummy plot showing hypothetical code-switch effect.
        fig, ax = plt.subplots(figsize=(8, 6))
        # Simulate data: WER vs % code-switch
        x = np.linspace(0, 60, 6)  # percentage code-switch
        y_ql = 12.8 + 0.2 * x  # simulated QuantoLens WER
        y_whisper = 14.6 + 0.5 * x  # simulated Whisper
        ax.plot(x, y_ql, 'o-', label='QuantoLens (simulated)')
        ax.plot(x, y_whisper, 's--', label='Whisper (simulated)')
        ax.set_xlabel('Code-Switch Percentage')
        ax.set_ylabel('WER (%)')
        ax.set_title('Simulated Code-Switch Performance')
        ax.legend()
        ax.grid(True, linestyle='--', alpha=0.3)
        plt.tight_layout()
        savefig(fig, '35_code_switch.pdf')
        plt.show()

    # -----------------------------------------------------------------
    # 14. Duration vs Accuracy (WER)
    # -----------------------------------------------------------------
    def plot_duration_vs_accuracy(self):
        if self.cv_features.empty or self.asr_results.empty:
            return
        merged = pd.merge(self.cv_features, self.asr_results, on='path', how='inner')
        if merged.empty:
            return

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.scatter(merged['duration_s'], merged['wer'], alpha=0.6)
        ax.set_xlabel('Duration (s)')
        ax.set_ylabel('WER')
        ax.set_title('WER vs Utterance Duration')
        # Regression
        mask = ~np.isnan(merged['duration_s']) & ~np.isnan(merged['wer'])
        if mask.sum() > 3:
            slope, intercept = np.polyfit(merged.loc[mask, 'duration_s'],
                                          merged.loc[mask, 'wer'], 1)
            x_vals = np.linspace(merged['duration_s'].min(), merged['duration_s'].max(), 100)
            ax.plot(x_vals, slope*x_vals + intercept, 'r--', label=f'slope={slope:.3f}')
            ax.legend()
        ax.grid(True, linestyle='--', alpha=0.3)
        plt.tight_layout()
        savefig(fig, '36_duration_vs_wer.pdf')
        plt.show()

    # -----------------------------------------------------------------
    # 15. CHiME-6 Session Activity Over Time
    # -----------------------------------------------------------------
    def plot_session_activity_chime(self):
        if self.chime_df is None or self.chime_df.empty:
            log.warning("CHiME-6 data missing; skipping session activity.")
            return

        df = self.chime_df
        if not {'start_time', 'end_time', 'speaker', '_session'}.issubset(df.columns):
            return

        # Convert times to seconds
        def to_seconds(t):
            try:
                parts = str(t).split(':')
                return sum(float(p) * 60 ** i for i, p in enumerate(reversed(parts)))
            except:
                return np.nan
        df['start_s'] = df['start_time'].apply(to_seconds)
        df['end_s'] = df['end_time'].apply(to_seconds)

        # Pick a session with many segments
        session_counts = df['_session'].value_counts()
        if session_counts.empty:
            return
        top_session = session_counts.idxmax()
        df_session = df[df['_session'] == top_session].sort_values('start_s')

        fig, ax = plt.subplots(figsize=(12, 6))
        # Plot speaker activity as horizontal bars
        speakers = df_session['speaker'].unique()
        colors = plt.cm.tab10(np.linspace(0, 1, len(speakers)))
        speaker_to_color = {sp: col for sp, col in zip(speakers, colors)}

        y_pos = 0
        for i, (idx, row) in enumerate(df_session.iterrows()):
            sp = row['speaker']
            start = row['start_s']
            end = row['end_s']
            if not np.isnan(start) and not np.isnan(end):
                ax.barh(sp, end - start, left=start, color=speaker_to_color.get(sp, 'gray'),
                        edgecolor='black', linewidth=0.5)
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Speaker')
        ax.set_title(f'Speaker Activity in Session: {top_session}')
        plt.tight_layout()
        savefig(fig, '37_chime_activity.pdf')
        plt.show()

    # -----------------------------------------------------------------
    # 16. Generate LaTeX Tables from Results
    # -----------------------------------------------------------------
    def generate_latex_tables(self):
        # Produce tables as strings, save to file
        tables = {}

        # Table 1: ASR performance summary
        if not self.asr_results.empty:
            summary = self.asr_results[['wer', 'cer']].describe().round(3)
            latex = summary.to_latex()
            tables['asr_summary'] = latex

        # Table 2: Acoustic features statistics
        if not self.cv_features.empty:
            feat_cols = ['duration_s', 'f0_mean', 'centroid_mean', 'bandwidth_mean',
                         'zcr_mean', 'rms_mean', 'voiced_fraction', 'speaking_rate']
            feat_cols = [c for c in feat_cols if c in self.cv_features.columns]
            if feat_cols:
                feat_stats = self.cv_features[feat_cols].describe().round(3)
                latex = feat_stats.to_latex()
                tables['acoustic_stats'] = latex

        # Table 3: MASSIVE intent distribution
        if self.massive_df is not None and not self.massive_df.empty and 'intent' in self.massive_df.columns:
            intent_counts = self.massive_df['intent'].value_counts().head(10)
            latex = intent_counts.to_frame('count').to_latex()
            tables['intent_counts'] = latex

        # Write to file
        if tables:
            with open(os.path.join(PATHS.output_dir, 'latex_tables.tex'), 'w', encoding='utf-8') as f:
                for name, latex in tables.items():
                    f.write(f'% Table: {name}\n')
                    f.write(latex)
                    f.write('\n\n')
            log.info(f"LaTeX tables saved to {PATHS.output_dir}/latex_tables.tex")

    # -----------------------------------------------------------------
    # 17. Additional: Error Analysis by Phoneme (if possible)
    # -----------------------------------------------------------------
    def plot_error_analysis_by_phoneme(self):
        # Placeholder: if we had phoneme alignments, we could plot error rates per phoneme.
        # For now, just skip.
        pass


# =====================================================================
# SECTION 13 — INTEGRATION INTO MAIN PIPELINE
# =====================================================================

def run_experimental_analysis(cv_features, asr_results, massive_df, chime_df):

    # Check that we have at least some data
    if cv_features.empty and asr_results.empty and massive_df is None and chime_df is None:
        log.warning("No data for experimental analysis.")
        return

    analyzer = ExperimentalAnalysis(cv_features, asr_results, massive_df, chime_df)
    analyzer.run_all()


# =====================================================================
# SECTION 14 — ADDITIONAL UTILITY: GENERATE COMPREHENSIVE REPORT
# =====================================================================

def generate_report(features_df, asr_df, massive_df, chime_df):

    lines = []
    lines.append("=" * 80)
    lines.append("EXPERIMENTAL REPORT")
    lines.append("=" * 80)

    # Common Voice
    if not features_df.empty:
        lines.append(f"Common Voice: {len(features_df)} clips processed.")
        lines.append(f"  Features: {', '.join(features_df.columns)}")
    else:
        lines.append("Common Voice: No data.")

    # ASR
    if not asr_df.empty:
        lines.append(f"ASR: {len(asr_df)} clips evaluated.")
        lines.append(f"  Mean WER: {asr_df['wer'].mean():.3f} ± {asr_df['wer'].std():.3f}")
        lines.append(f"  Mean CER: {asr_df['cer'].mean():.3f} ± {asr_df['cer'].std():.3f}")
    else:
        lines.append("ASR: No results.")

    # MASSIVE
    if massive_df is not None and not massive_df.empty:
        lines.append(f"MASSIVE: {len(massive_df)} utterances.")
        if 'intent' in massive_df.columns:
            lines.append(f"  Unique intents: {massive_df['intent'].nunique()}")
    else:
        lines.append("MASSIVE: No data.")

    # CHiME-6
    if chime_df is not None and not chime_df.empty:
        lines.append(f"CHiME-6: {len(chime_df)} segments.")
        if 'speaker' in chime_df.columns:
            lines.append(f"  Unique speakers: {chime_df['speaker'].nunique()}")
    else:
        lines.append("CHiME-6: No data.")

    lines.append("=" * 80)

    report = "\n".join(lines)
    with open(os.path.join(PATHS.output_dir, 'report.txt'), 'w', encoding='utf-8') as f:
        f.write(report)
    log.info(f"Report saved to {PATHS.output_dir}/report.txt")
    print(report)




def main():
    log.info("=== Starting pipeline ===")

    # --- Common Voice ---
    cv = CommonVoiceCorpus(PATHS.cv_root)
    cv.load_tsvs()
    print(cv.summarize())
    cv.plot_metadata_overview()

    sample_paths = cv.sample_clip_paths(n=50)

    extractor = AdvancedAudioFeatureExtractor()
    cv_features = pd.DataFrame()
    if sample_paths:
        extractor.plot_single_clip(sample_paths[0])
        cv_features = extractor.batch_extract(sample_paths)
        extractor.plot_feature_distributions(cv_features)
        extractor.plot_feature_correlations(cv_features)
        cv_features.to_csv(os.path.join(PATHS.output_dir, "cv_audio_features.csv"), index=False)

    # --- ASR baseline ---
    asr_results = pd.DataFrame()
    if sample_paths:
        asr = ASRBaseline(model_size="small", language="as")
        asr_results = asr.evaluate(cv, sample_paths)
        asr.plot_results(asr_results)
        if not asr_results.empty:
            asr_results.to_csv(os.path.join(PATHS.output_dir, "asr_results.csv"), index=False)

    # --- Unrated transcripts ---
    unrated = UnratedTranscripts(PATHS.unrated_root)
    unrated.scan()
    unrated.preview(n=2)
    stats_df = unrated.basic_stats()
    unrated.plot_file_sizes(stats_df)

    # --- MASSIVE ---
    massive = MassiveCorpus(PATHS.massive_root)
    massive.load()
    massive.plot_overview()

    # --- CHiME-6 ---
    chime = Chime6Corpus(PATHS.chime_trans, PATHS.chime_floor)
    chime.load_transcriptions()
    chime.plot_speaker_activity()
    chime.list_floorplans()

    # --- Statistical Analysis ---
    if not cv_features.empty:
        stats_analyzer = StatisticalAnalyzer(cv_features)
        stats_analyzer.plot_feature_pairplot()
        if "gender" in cv_features.columns:
            stats_analyzer.perform_anova(group_col="gender")
        else:
            log.info("No gender column; skipping ANOVA.")

    # --- Machine Learning ---
    if not cv_features.empty:
        ml = MLModel(cv_features)
        if ml.prepare_data(target_col="gender"):
            ml.train_evaluate()
        else:
            log.warning("ML preparation failed.")

    # --- Quantum-inspired feature mapping ---
    if not cv_features.empty:
        numeric_cols = cv_features.select_dtypes(include=np.number).columns
        numeric_cols = [c for c in numeric_cols if c not in ["duration_s", "path"]]
        if len(numeric_cols) >= 2:
            X = cv_features[numeric_cols].dropna().values
            if X.shape[0] > 10 and X.shape[1] > 1:
                qmapper = QuantumFeatureMapper(n_components=min(8, X.shape[1]))
                X_q = qmapper.fit_transform(X)
                labels = None
                if "gender" in cv_features.columns:
                    labels = cv_features.loc[cv_features[numeric_cols].dropna().index, "gender"].values
                qmapper.plot_quantum_features(X_q, labels)
                np.save(os.path.join(PATHS.output_dir, "quantum_features.npy"), X_q)

    # --- Advanced visualization ---
    if not cv_features.empty:
        AdvancedVisualizer.plot_all_acoustic_aggregates(cv_features)

    # --- Cross-dataset summary ---
    cross_dataset_summary(cv_features, asr_results)

    # =================================================================
    # NEW: Experimental Analysis (integrated here)
    # =================================================================
    if not cv_features.empty or not asr_results.empty or massive.df is not None or chime.segments_df is not None:
        run_experimental_analysis(
            cv_features=cv_features,
            asr_results=asr_results,
            massive_df=massive.df if massive.df is not None else None,
            chime_df=chime.segments_df if chime.segments_df is not None else None
        )

    # Generate report
    generate_report(cv_features, asr_results,
                    massive.df if massive.df is not None else None,
                    chime.segments_df if chime.segments_df is not None else None)

    log.info("=== Pipeline complete. Check the pipeline_outputs/ folder. ===")

if __name__ == "__main__":
    main()